In [1]:

from dill.pointers import children
from sympy import print_rcode

from praca_magisterska.v2.ContextsAndLP import LittleProblem
from praca_magisterska.v2.TermAndFormulas import *
from praca_magisterska.v2.ContextsAndLP import *
from praca_magisterska.v2.HelpfullFunctions import *
import re
from typing import List


1


In [2]:
import sys
from pathlib import Path

# adjust if your path differs
PROJECT_ROOT = Path.home() / "PycharmProjects" / "Magisterka"
PKG_ROOT = PROJECT_ROOT / "nanoGPT-20251228T135841Z-3-001" / "nanoGPT"

sys.path.insert(0, str(PKG_ROOT))


class Rule:
    @abstractmethod
    def __init__(self,a,b):
        idx = -1
        pass
    @abstractmethod
    def bottom_up(self ,*args):
        pass
    @abstractmethod
    def top_down(self ,*args):
        pass
    def __str__(self):
        return self.__class__.__name__
    def __repr__(self):
        return self.__class__.__name__



In [3]:
class Assumption(Rule):
    def __init__(self):
        self.idx = 0
    def top_down(self,x,phi = Truth(),BaseContext = []):
        if not x == []:
            raise TypeError("Bad arguments")
        if not isinstance(phi,Formula):
            raise TypeError("Bad arguments")
        if not isinstance(BaseContext,list):
            raise TypeError("Bad arguments")
        for i in BaseContext:
            if not isinstance(i, Formula):
                raise TypeError("Bad arguments")
        Left = Context(BaseContext , [phi])
        Right = phi
        return LittleProblem(Left,Right)
    def bottom_up(self ,x):
        if not isinstance(x,LittleProblem):
            raise TypeError("Bad arguments")
        phi = x.conclusion
        if x.assumptions.additional_context.__len__() == 0:
            if not phi in x.assumptions.BaseContext :
                raise TypeError("Bad arguments")
        elif x.assumptions.additional_context.__len__() == 1:
            if not phi in x.assumptions.additional_context:
                raise TypeError("Bad arguments")
        else:
            raise TypeError("Bad arguments")
        return []

In [4]:
class Weakening(Rule):
    def __init__(self):
        self.idx = 1
    def top_down(self, x, psi=Truth()):
        if not isinstance(x, list):
            raise TypeError("Bad arguments")
        if not x.__len__() == 1:
            raise TypeError("Bad arguments")
        lp = x[0]
        if not isinstance(lp, LittleProblem):
            raise TypeError("Bad arguments")
        if not isinstance(psi, Formula):
            raise TypeError("Bad arguments")
        Left = lp.assumptions + psi
        Right = psi
        return LittleProblem(Left, Right)

    def bottom_up(self, x,psi = Truth()):
        if not isinstance(x, LittleProblem):
            raise TypeError("Bad arguments")
        if not isinstance(psi, Formula):
            raise TypeError("Bad arguments")
        if not psi in x.assumptions:
            raise TypeError("Bad arguments")
        Left = x.assumptions - psi
        Right = x.conclusion
        return [LittleProblem(Left, Right)]


In [5]:
class ImplicationIntroduction(Rule):
    def __init__(self):
        self.idx = 2
    def top_down(self,x,phi = Truth):
        if not isinstance(x, list):
            raise TypeError("Bad arguments")
        if x.__len__() != 1:
            raise TypeError("Bad arguments")
        if not isinstance(x[0],LittleProblem):
            raise TypeError("Bad arguments")
        if not phi in x[0].assumptions:
            raise TypeError("Bad arguments")
        psi = x[0].conclusion
        Gamma = x[0].assumptions - phi
        return LittleProblem(Gamma, Implication(phi,psi))
    def bottom_up(self ,x):
        if not isinstance(x,LittleProblem):
            raise TypeError("Bad arguments")
        if isinstance(x.conclusion,Implication):
            ans_conclusion = x.conclusion.Right()
            ans_assumptions = x.assumptions + x.conclusion.Left()
            return [LittleProblem(ans_assumptions,ans_conclusion)]
        else:
            raise TypeError("Bad arguments")

In [6]:
# ... existing code ...
class NegationIntroduction(Rule):
    def __init__(self):
        self.idx = 3

    def top_down(self, x, phi = Truth()):
        if not isinstance(x, list):
            raise TypeError("Bad arguments")
        if x.__len__() != 1:
            raise TypeError("Bad arguments")
        if not isinstance(x[0], LittleProblem):
            raise TypeError("Bad arguments")
        if not isinstance(phi, Formula):
            raise TypeError("Bad arguments")
        if not isinstance(x[0].conclusion, Lie):
            raise TypeError("Bad arguments")
        if not phi in x[0].assumptions:
            raise TypeError("Bad arguments")
        Gamma = x[0].assumptions - phi
        return LittleProblem(Gamma, Negation(phi))

    def bottom_up(self, x):
        if not isinstance(x, LittleProblem):
            raise TypeError("Bad arguments")
        if not isinstance(x.conclusion, Negation):
            raise TypeError("Bad arguments")
        phi = x.conclusion.Left()
        Gamma = x.assumptions
        return [LittleProblem(Gamma + phi, Lie())]

In [7]:
class ImplicationElimination(Rule):  # (→E)
    def __init__(self):
        self.idx = 4
    def top_down(self,x):
        if not isinstance(x,list):
            raise TypeError("Bad arguments")
        if x.__len__() != 2:
            raise TypeError("Bad arguments")
        if not (isinstance(x[0],LittleProblem) and isinstance(x[1],LittleProblem)):
            raise TypeError("Bad arguments")
        if  set(x[0].assumptions.base_context) != set(x[1].assumptions.base_context):
            raise TypeError("Bad arguments")
        if not isinstance(x[0].conclusion,Implication):
            raise TypeError("Bad arguments")
        if x[0].conclusion.Left() != x[1].conclusion:
            raise TypeError("Bad arguments")
        Gamma1 = x[0].assumptions
        Gamma2 = x[1].assumptions
        psi = x[0].conclusion.Right()
        return LittleProblem(Gamma1 + Gamma2 , psi)
    def bottom_up(self, x,Gamma1 = Context([],[]),Gamma2= Context([],[]),phi = Truth()):
        if not isinstance(x, LittleProblem):
            raise TypeError("Bad arguments")
        if not isinstance(Gamma1, Context):
            raise TypeError("Bad arguments")
        if not isinstance(Gamma2, Context):
            raise TypeError("Bad arguments")
        if not isinstance(phi, Formula):
            raise TypeError("Bad arguments")
        if not Gamma1 + Gamma2 == x.assumptions:
            raise TypeError("Bad arguments")
        psi = x.conclusion
        First = LittleProblem(Gamma1,Implication(phi,psi))
        Second = LittleProblem(Gamma2,phi)
        return [First,Second]

In [8]:
class NegationElimination(Rule):
    def __init__(self):
        self.idx = 5
    def top_down(self , x):
        if not isinstance(x, list):
            raise TypeError("Bad arguments")
        if x.__len__() != 2:
            raise TypeError("Bad arguments")
        if not isinstance(x[0], LittleProblem) and isinstance(x[1], LittleProblem):
            raise TypeError("Bad arguments")
        if  set(x[0].assumptions.base_context) != set(x[1].assumptions.base_context):
            raise TypeError("Bad arguments")
        if not isinstance(x[0].conclusion, Negation):
            raise TypeError("Bad arguments")
        phi = x[0].conclusion.Left()
        if x[1].conclusion != phi:
            raise TypeError("Bad arguments")
        Gamma1 = x[0].assumptions
        Gamma2 = x[1].assumptions
        return LittleProblem(Gamma1 + Gamma2 , Lie())
    def bottom_up(self ,x,Gamma1 = Context([],[]),Gamma2= Context([],[]),phi = Truth()):
        if not isinstance(x, LittleProblem):
            raise TypeError("Bad arguments")
        if not isinstance(Gamma1, Context):
            raise TypeError("Bad arguments")
        if not isinstance(Gamma2, Context):
            raise TypeError("Bad arguments")
        if not isinstance(phi, Formula):
            raise TypeError("Bad arguments")
        if not Gamma1 + Gamma2 == x.assumptions:
            raise TypeError("Bad arguments")
        if not x.conclusion == Lie():
            raise TypeError("Bad arguments")
        First = LittleProblem(Gamma1,Negation(phi))
        Second = LittleProblem(Gamma2,phi)
        return [First,Second]

In [9]:
class ConjunctionIntroduction(Rule):
    def __init__(self):
        self.idx = 6
    def top_down(self , x):
        if not isinstance(x, list):
            raise TypeError("Bad arguments")
        if x.__len__() != 2:
            raise TypeError("Bad arguments")
        if not isinstance(x[0], LittleProblem) and isinstance(x[1], LittleProblem):
            raise TypeError("Bad arguments")
        if  set(x[0].assumptions.base_context) != set(x[1].assumptions.base_context):
            raise TypeError("Bad arguments")
        Gamma1 = x[0].assumptions
        Gamma2 = x[1].assumptions
        phi = x[0].conclusion
        psi = x[1].conclusion
        return LittleProblem(Gamma1 + Gamma2 , Conjunction(phi,psi))
    def bottom_up(self ,x,Gamma1 = Context([],[]),Gamma2= Context([],[])):
        if not isinstance(x, LittleProblem):
            raise TypeError("Bad arguments")
        if not isinstance(Gamma1, Context):
            raise TypeError("Bad arguments")
        if not isinstance(Gamma2, Context):
            raise TypeError("Bad arguments")
        if not Gamma1 + Gamma2 == x.assumptions:
            raise TypeError("Bad arguments")
        if not isinstance(x.conclusion,Conjunction):
            raise TypeError("Bad arguments")
        phi = x.conclusion.Left()
        psi = x.conclusion.Right()
        First = LittleProblem(Gamma1,phi)
        Second = LittleProblem(Gamma2,psi)
        return [First,Second]

In [10]:
class DisjunctionIntroduction1(Rule):
    def __init__(self):
        self.idx = 7
    def top_down(self, x, psi=Truth()):
        if not isinstance(x, list):
            raise TypeError("Bad arguments")
        if not x.__len__() == 1:
            raise TypeError("Bad arguments")
        lp = x[0]
        if not isinstance(lp, LittleProblem):
            raise TypeError("Bad arguments")
        if not isinstance(psi, Formula):
            raise TypeError("Bad arguments")
        Left = lp.assumptions
        Right = Disjunction(lp.conclusion,psi)
        return LittleProblem(Left, Right)
    def bottom_up(self, x):
        if not isinstance(x, LittleProblem):
            raise TypeError("Bad arguments")
        if not isinstance(x.conclusion, Disjunction):
            raise TypeError("Bad arguments")
        Left = x.assumptions
        Right = x.conclusion.Left()
        return [LittleProblem(Left, Right)]

In [11]:
class DisjunctionIntroduction2(Rule):
    def __init__(self):
        self.idx = 8
    def top_down(self, x, phi=Truth()):
        if not isinstance(x, list):
            raise TypeError("Bad arguments")
        if not x.__len__() == 1:
            raise TypeError("Bad arguments")
        lp = x[0]
        if not isinstance(lp, LittleProblem):
            raise TypeError("Bad arguments")
        if not isinstance(phi, Formula):
            raise TypeError("Bad arguments")
        Left = lp.assumptions
        Right = Disjunction(phi,lp.conclusion)
        return LittleProblem(Left, Right)
    def bottom_up(self, x):
        if not isinstance(x, LittleProblem):
            raise TypeError("Bad arguments")
        if not isinstance(x.conclusion, Disjunction):
            raise TypeError("Bad arguments")
        Left = x.assumptions
        Right = x.conclusion.Right()
        return [LittleProblem(Left, Right)]

In [12]:
class TruthIntroduction(Rule):
    def __init__(self):
        self.idx = 9
    def top_down(self,x,BaseContext = []):
        if not x == []:
            raise TypeError("Bad arguments")
        if not isinstance(BaseContext,list):
            raise TypeError("Bad arguments")
        for i in BaseContext:
            if not isinstance(i, Formula):
                raise TypeError("Bad arguments")
        Left = Context(BaseContext ,[])
        Right = Truth()
        return LittleProblem(Left,Right)
    def bottom_up(self ,x):
        if not isinstance(x,LittleProblem):
            raise TypeError("Bad arguments")
        if not x.assumptions.additional_context == []:
            raise TypeError("Bad arguments")
        if not x.conclusion == Truth():
            raise TypeError("Bad arguments")
        return []

In [13]:
class ConjunctionElimination1(Rule):
    def __init__(self):
        self.idx = 10
    def top_down(self ,x):
        if not isinstance(x,list):
            raise TypeError("Bad arguments")
        if x.__len__() != 1:
            raise TypeError("Bad arguments")
        if not isinstance(x[0],LittleProblem):
            raise TypeError("Bad arguments")
        if not isinstance(x[0].conclusion,Conjunction):
            raise TypeError("Bad arguments")
        phi = x[0].conclusion.Left()
        Gamma = x[0].assumptions
        return LittleProblem(Gamma , phi)
    def bottom_up(self,x,psi  = Truth()):
        if not isinstance(x, LittleProblem):
            raise TypeError("Bad arguments")
        if not isinstance(psi, Formula):
            raise TypeError("Bad arguments")
        phi = x.conclusion
        Gamma = x.assumptions
        return [LittleProblem(Gamma,Conjunction(phi,psi))]


In [14]:
class ConjunctionElimination2(Rule):
    def __init__(self):
        self.idx = 11
    def top_down(self ,x):
        if not isinstance(x,list):
            raise TypeError("Bad arguments")
        if x.__len__() != 1:
            raise TypeError("Bad arguments")
        if not isinstance(x[0],LittleProblem):
            raise TypeError("Bad arguments")
        if not isinstance(x[0].conclusion,Conjunction):
            raise TypeError("Bad arguments")
        psi = x[0].conclusion.Right()
        Gamma = x[0].assumptions
        return LittleProblem(Gamma , psi)
    def bottom_up(self,x,phi  = Truth()):
        if not isinstance(x, LittleProblem):
            raise TypeError("Bad arguments")
        if not isinstance(phi, Formula):
            raise TypeError("Bad arguments")
        psi = x.conclusion
        Gamma = x.assumptions
        return [LittleProblem(Gamma,Conjunction(phi,psi))]

In [15]:
class DisjunctionElimination(Rule):
    def __init__(self):
        self.idx = 12
    def top_down(self ,x):
        if not isinstance(x,list):
            raise TypeError("Bad arguments")
        if x.__len__() != 3:
            raise TypeError("Bad arguments")
        if not (isinstance(x[0],LittleProblem) and isinstance(x[1],LittleProblem)) and isinstance(x[2],LittleProblem):
            raise TypeError("Bad arguments")
        if  set(x[0].assumptions.base_context) != set(x[1].assumptions.base_context):
            raise TypeError("Bad arguments")
        if  set(x[0].assumptions.base_context) != set(x[2].assumptions.base_context):
            raise TypeError("Bad arguments")
        if  set(x[1].assumptions.base_context) != set(x[2].assumptions.base_context):
            raise TypeError("Bad arguments")
        if not isinstance(x[0].conclusion,Disjunction):
            raise TypeError("Bad arguments")
        phi = x[0].conclusion.Left()
        psi = x[0].conclusion.Right()
        if not phi in x[1].assumptions:
            raise TypeError("Bad arguments")
        if not psi in x[2].assumptions:
            raise TypeError("Bad arguments")
        Gamma1 = x[0].assumptions
        Gamma2 = x[1].assumptions - phi
        Gamma3 = x[2].assumptions - psi
        if not x[1].conclusion == x[2].conclusion:
            raise TypeError("Bad arguments")
        rho = x[1].conclusion
        return LittleProblem(Gamma1 + Gamma2 + Gamma3 , rho)
    def bottom_up(self,x,Gamma1 = Context([],[]),Gamma2= Context([],[]),Gamma3= Context([],[]),phi = Truth(),psi = Truth()):
        if not isinstance(x, LittleProblem):
            raise TypeError("Bad arguments")
        if not isinstance(Gamma1, Context):
            raise TypeError("Bad arguments")
        if not isinstance(Gamma2, Context):
            raise TypeError("Bad arguments")
        if not isinstance(Gamma3, Context):
            raise TypeError("Bad arguments")
        if not Gamma1 + Gamma2 + Gamma3 == x.assumptions:
            raise TypeError("Bad arguments")
        rho = x.conclusion
        First  = LittleProblem(Gamma1,Disjunction(phi,psi))
        Second = LittleProblem(Gamma2+phi,rho)
        Third  = LittleProblem(Gamma3+psi,rho)
        return [First,Second,Third]

In [16]:
class LieElimination(Rule):
    def __init__(self):
        self.idx = 13
    def top_down(self,x,phi = Truth()):
        if not isinstance(x,list):
            raise TypeError("Bad arguments")
        if not x.__len__() == 1:
            raise TypeError("Bad arguments")
        if not isinstance(x[0],LittleProblem):
            raise TypeError("Bad arguments")
        if not isinstance(x[0].conclusion,Lie):
            raise TypeError("Bad arguments")
        if not isinstance(phi, Formula):
            raise TypeError("Bad arguments")
        Gamma = x[0].assumptions
        return LittleProblem(Gamma, phi)
    def bottom_up(self ,x):
        if not isinstance(x,LittleProblem):
            raise TypeError("Bad arguments")
        return [LittleProblem(x.assumptions,Lie())]

In [17]:
class IffIntroduction(Rule):
    def __init__(self):
        self.idx = 14
    def top_down(self,x):
        if not isinstance(x,list):
            raise TypeError("Bad arguments")
        if not x.__len__() == 2:
            raise TypeError("Bad arguments")
        if not (isinstance(x[0],LittleProblem) and isinstance(x[1],LittleProblem)):
            raise TypeError("Bad arguments")
        if  set(x[0].assumptions.base_context) != set(x[1].assumptions.base_context):
            raise TypeError("Bad arguments")
        psi = x[0].conclusion
        phi = x[1].conclusion
        if not (psi in x[1].assumptions and phi in x[0].assumptions):
            raise TypeError("Bad arguments")
        Gamma1 = x[0].assumptions - psi
        Gamma2 = x[1].assumptions - phi
        return LittleProblem(Gamma1 + Gamma2 , Iff(phi,psi))
    def bottom_up(self,x, Gamma1 = Context([],[]),Gamma2= Context([],[])):
        if not isinstance(x, LittleProblem):
            raise TypeError("Bad arguments")
        if not isinstance(Gamma1, Context):
            raise TypeError("Bad arguments")
        if not isinstance(Gamma2, Context):
            raise TypeError("Bad arguments")
        if not isinstance(x.conclusion,Iff):
            raise TypeError("Bad arguments")
        phi = x.conclusion.Left()
        psi = x.conclusion.Right()
        if not Gamma1 + Gamma2== x.assumptions:
            raise TypeError("Bad arguments")
        First = LittleProblem(Gamma1+phi,psi)
        Second = LittleProblem(Gamma2+psi,phi)
        return [First,Second]

In [18]:
class IffElimination1(Rule):  # (→E)
    def __init__(self):
        self.idx = 15
    def top_down(self,x):
        if not isinstance(x,list):
            raise TypeError("Bad arguments")
        if x.__len__() != 2:
            raise TypeError("Bad arguments")
        if not (isinstance(x[0],LittleProblem) and isinstance(x[1],LittleProblem)):
            raise TypeError("Bad arguments")
        if  set(x[0].assumptions.base_context) != set(x[1].assumptions.base_context):
            raise TypeError("Bad arguments")
        if not isinstance(x[0].conclusion,Iff):
            raise TypeError("Bad arguments")
        if x[0].conclusion.Left() != x[1].conclusion:
            raise TypeError("Bad arguments")
        Gamma1 = x[0].assumptions
        Gamma2 = x[1].assumptions
        psi = x[0].conclusion.Right()
        return LittleProblem(Gamma1 + Gamma2 , psi)
    def bottom_up(self, x,Gamma1 = Context([],[]),Gamma2= Context([],[]),phi = Truth()):
        if not isinstance(x, LittleProblem):
            raise TypeError("Bad arguments")
        if not isinstance(Gamma1, Context):
            raise TypeError("Bad arguments")
        if not isinstance(Gamma2, Context):
            raise TypeError("Bad arguments")
        if not isinstance(phi, Formula):
            raise TypeError("Bad arguments")
        if not Gamma1 + Gamma2 == x.assumptions:
            raise TypeError("Bad arguments")
        psi = x.conclusion
        First = LittleProblem(Gamma1,Iff(phi,psi))
        Second = LittleProblem(Gamma2,phi)
        return [First,Second]

In [19]:
class IffElimination2(Rule):  # (→E)
    def __init__(self):
        self.idx = 16
    def top_down(self,x):
        if not isinstance(x,list):
            raise TypeError("Bad arguments")
        if x.__len__() != 2:
            raise TypeError("Bad arguments")
        if not (isinstance(x[0],LittleProblem) and isinstance(x[1],LittleProblem)):
            raise TypeError("Bad arguments")
        if  set(x[0].assumptions.base_context) != set(x[1].assumptions.base_context):
            raise TypeError("Bad arguments")
        if not isinstance(x[0].conclusion,Iff):
            raise TypeError("Bad arguments")
        if x[0].conclusion.Right() != x[1].conclusion:
            raise TypeError("Bad arguments")
        Gamma1 = x[0].assumptions
        Gamma2 = x[1].assumptions
        phi = x[0].conclusion.Left()
        return LittleProblem(Gamma1 + Gamma2 , phi)
    def bottom_up(self, x,Gamma1 = Context([],[]),Gamma2= Context([],[]),psi = Truth()):
        if not isinstance(x, LittleProblem):
            raise TypeError("Bad arguments")
        if not isinstance(Gamma1, Context):
            raise TypeError("Bad arguments")
        if not isinstance(Gamma2, Context):
            raise TypeError("Bad arguments")
        if not isinstance(psi, Formula):
            raise TypeError("Bad arguments")
        if not Gamma1 + Gamma2 == x.assumptions:
            raise TypeError("Bad arguments")
        phi = x.conclusion
        First = LittleProblem(Gamma1,Iff(phi,psi))
        Second = LittleProblem(Gamma2,psi)
        return [First,Second]

In [20]:
class RAA(Rule):
    def __init__(self):
        self.idx = 17
    def top_down(self,x,phi = Truth()):
        if not isinstance(x,list):
            raise TypeError("Bad arguments")
        if x.__len__() != 1:
            raise TypeError("Bad arguments")
        if not isinstance(x[0],LittleProblem):
            raise TypeError("Bad arguments")
        if not isinstance(x[0].conclusion,Lie):
            raise TypeError("Bad arguments")
        if not isinstance(phi, Formula):
            raise TypeError("Bad arguments")
        if not Negation(phi) in x[0].assumptions:
            raise TypeError("Bad arguments")
        Gamma = x[0].assumptions - Negation(phi)
        return LittleProblem(Gamma,phi)
    def bottom_up(self,x):
        if not isinstance(x, LittleProblem):
            raise TypeError("Bad arguments")
        phi = x.conclusion
        Gamma = x.assumptions
        Left = Gamma + Negation(phi)
        Right = Lie()
        return [LittleProblem(Left,Right)]


In [21]:
class NegationOfNegation(Rule):
    def __init__(self):
        self.idx = 18
    def top_down(self,x):
        if not isinstance(x,list):
            raise TypeError("Bad arguments")
        if x.__len__() != 1:
            raise TypeError("Bad arguments")
        if not isinstance(x[0],LittleProblem):
            raise TypeError("Bad arguments")
        if not isinstance(x[0].conclusion,Negation):
            raise TypeError("Bad arguments")
        if not isinstance(x[0].conclusion.Left(),Negation):
            raise TypeError("Bad arguments")
        phi = x[0].conclusion.Left().Left()
        return LittleProblem(x[0].assumptions,phi)
    def bottom_up(self,x):
        if not isinstance(x, LittleProblem):
            raise TypeError("Bad arguments")
        phi = x.conclusion
        return [LittleProblem(x.assumptions,Negation(Negation(phi)))]

In [22]:
class TND(Rule):
    def __init__(self):
        self.idx = 19
    def top_down(self ,x,phi = Truth(),BaseContext = []):
        if not x == []:
            raise TypeError("Bad arguments")
        if not isinstance(BaseContext,list):
            raise TypeError("Bad arguments")
        for i in BaseContext:
            if not isinstance(i, Formula):
                raise TypeError("Bad arguments")
        if not isinstance(phi, Formula):
            raise TypeError("Bad arguments")
        Right = Disjunction(phi,Negation(phi))
        Left = Context(BaseContext,[])
        return LittleProblem(Left,Right)
    def bottom_up(self,x):
        if not isinstance(x, LittleProblem):
            raise TypeError("Bad arguments")
        if not isinstance(x.conclusion,Disjunction):
            raise TypeError("Bad arguments")
        if not Negation(x.conclusion.Left()) == x.conclusion.Right():
            raise TypeError("Bad arguments")
        if not x.assumptions.additional_context == []:
            raise TypeError("Bad arguments")
        return []

In [23]:
def to_infix_LP(LP: LittleProblem) -> str:
    ans = ""
    for i in LP.assumptions.base_context:
        ans += to_infix(i) + ", "
    ans = ans[:-2]
    ans += " ; "
    for i in LP.assumptions.additional_context:
        ans += to_infix(i) + ", "
    ans = ans[:-2]
    ans += " ⊢ " + to_infix(LP.conclusion)
    return ans


class structure_of_line_top_down:
    def __init__(self,Key: int, args_keys: list[int], rule: Rule,Proof_so_far:dict,formula_text : str = "",BaseContext: list = []):
        self.Key = Key
        self.args_keys = args_keys
        self.rule = rule
        match rule:
            case Assumption():
                phi = parse_infix(formula_text)
                self.LP = rule.top_down([],phi = phi, BaseContext = BaseContext)

            case Weakening():
                phi = parse_infix(formula_text)
                x = [Proof_so_far[args_keys[0]].LP]
                self.LP = rule.top_down(x,phi)

            case ImplicationIntroduction():
                phi = Proof_so_far[args_keys[0]].LP.conclusion
                x = [Proof_so_far[args_keys[1]].LP]
                self.LP = rule.top_down(x,phi=phi)

            case NegationIntroduction():
                x = [Proof_so_far[args_keys[1]].LP]
                phi = Proof_so_far[args_keys[0]].LP.conclusion
                self.LP = rule.top_down(x,phi=phi)

            case ImplicationElimination():
                x = [Proof_so_far[args_keys[0]].LP,Proof_so_far[args_keys[1]].LP]
                self.LP = rule.top_down(x)

            case NegationElimination():
                x = [Proof_so_far[args_keys[0]].LP,Proof_so_far[args_keys[1]].LP]
                self.LP = rule.top_down(x)

            case ConjunctionIntroduction():
                x = [Proof_so_far[args_keys[0]].LP,Proof_so_far[args_keys[1]].LP]
                self.LP = rule.top_down(x)

            case DisjunctionIntroduction1():
                psi = parse_infix(formula_text).Right()
                x = [Proof_so_far[args_keys[0]].LP]
                self.LP = rule.top_down(x,psi=psi)

            case DisjunctionIntroduction2():
                phi = parse_infix(formula_text).Left()
                x = [Proof_so_far[args_keys[0]].LP]
                self.LP = rule.top_down(x,phi=phi)

            case TruthIntroduction():
                x = []
                self.LP = rule.top_down(x,BaseContext = BaseContext)

            case ConjunctionElimination1():
                x = [Proof_so_far[args_keys[0]].LP]
                self.LP = rule.top_down(x)

            case ConjunctionElimination2():
                x = [Proof_so_far[args_keys[0]].LP]
                self.LP = rule.top_down(x)

            case DisjunctionElimination():
                x = [Proof_so_far[args_keys[0]].LP ,Proof_so_far[args_keys[1]].LP ,Proof_so_far[args_keys[2]].LP]
                self.LP = rule.top_down(x)

            case LieElimination():
                x = [Proof_so_far[args_keys[0]].LP]
                phi = parse_infix(formula_text)
                self.LP = rule.top_down(x,phi=phi)

            case IffIntroduction():
                x = [Proof_so_far[args_keys[0]].LP,Proof_so_far[args_keys[1]].LP]
                self.LP = rule.top_down(x)

            case IffElimination1():
                x = [Proof_so_far[args_keys[0]].LP,Proof_so_far[args_keys[1]].LP]
                #print(rule.top_down(x),"----")
                self.LP = rule.top_down(x)

            case IffElimination2():
                x = [Proof_so_far[args_keys[0]].LP,Proof_so_far[args_keys[1]].LP]
                self.LP = rule.top_down(x)

            case RAA():
                phi_neg = Proof_so_far[args_keys[0]].LP.conclusion
                if not isinstance(phi_neg,Negation):
                    raise TypeError("Bad arguments")
                phi = phi_neg.Left()
                x = [Proof_so_far[args_keys[1]].LP]
                self.LP = rule.top_down(x,phi=phi)

            case NegationOfNegation():
                x = [Proof_so_far[args_keys[0]].LP]
                self.LP = rule.top_down(x)

            case TND():
                x = []
                phi = parse_infix(formula_text).Left()
                self.LP = rule.top_down(x,phi=phi,BaseContext = BaseContext)
            case _:
                raise TypeError("Bad rule")


    def __str__(self):
        return str(self.Key) +" "+ str(self.args_keys) + " "+to_infix_LP(self.LP) +" "+ str(self.rule)
    def __repr__(self):
        return repr(self.Key) +" "+ repr(self.args_keys) +" "+ repr(self.LP) +" "+ repr(self.rule)


In [24]:
proof = dict()
l1 = structure_of_line_top_down(1,[],Assumption(),dict(),formula_text="p ∨ q")
proof[l1.Key] = l1
l2 = structure_of_line_top_down(2,[1],Weakening(),proof,formula_text="p")

proof[l2.Key] = l2
print(proof)

{1: 1 []  ; R0(p) ∨ R0(q) ⊢ R0(p) ∨ R0(q) Assumption, 2: 2 [1]  ; R0(p), R0(p) ∨ R0(q) ⊢ R0(p) Weakening}


In [25]:
def parseProof(proof,base_context = []):
    text_splited = []
    proof = proof.split("\n")
    for i in proof:
        i_splited = i.split(".")
        i_splited = [i_splited[0]] + i_splited[1].split("  ")
        i_splited = list(filter(lambda x: x!= '',i_splited))
        ii = i_splited[:2]
        for I in i_splited[2:]:
            ii += i_splited[2].split(",")
        i_splited = list(filter(lambda x: x!= '',i_splited))
        i_splited_final = []
        for j in i_splited:
            i_splited_final += j.split("–")
        i_splited = i_splited_final
        i_splited_final = []
        for j in i_splited:
            i_splited_final += j.split(",")
        i_splited_final = list(filter(lambda x: x!= '',i_splited_final))
        for I in range(i_splited_final.__len__()):
            i_splited_final[I] = i_splited_final[I].replace(" ", "")
            i_splited_final[I] = i_splited_final[I].replace(",","")
        text_splited.append(i_splited_final)
    for i in range(len(text_splited)):
        idxs = []
        for j in range(len(text_splited[i])):
            if j == 0:
                text_splited[i][j] = int(text_splited[i][j])
            elif j == 1:
                pass
            elif j == 2:
                text_splited[i][j] = text_splited[i][j].replace(" ", "")
                match text_splited[i][j]:
                    case 'assumption':
                        text_splited[i][j] = Assumption()
                    case "weakening":
                        text_splited[i][j] = Weakening()
                    case "→-introduction":
                        text_splited[i][j] = ImplicationIntroduction()
                    case "¬-introduction":
                        text_splited[i][j] = NegationIntroduction()
                    case "→-elimination":
                        text_splited[i][j] = ImplicationElimination()
                    case "¬-elimination":
                        text_splited[i][j] = NegationElimination()
                    case "∧-introduction":
                        text_splited[i][j] = ConjunctionIntroduction()
                    case "∨-introduction1":
                        text_splited[i][j] = DisjunctionIntroduction1()
                    case "∨-introduction2":
                        text_splited[i][j] = DisjunctionIntroduction2()
                    case "⊤-introduction":
                        text_splited[i][j] = TruthIntroduction()
                    case "∧-elimination1":
                        text_splited[i][j] = ConjunctionElimination1()
                    case "∧-elimination2":
                        text_splited[i][j] = ConjunctionElimination2()
                    case "∨-elimination":
                        text_splited[i][j] = DisjunctionElimination()
                    case "⊥-elimination":
                        text_splited[i][j] = LieElimination()
                    case "↔-introduction":
                        text_splited[i][j] = IffIntroduction()
                    case "↔-elimination1":
                        text_splited[i][j] = IffElimination1()
                    case "↔-elimination2":
                        text_splited[i][j] = IffElimination2()
                    case "RAA":
                        text_splited[i][j] = RAA()
                    case "¬¬-elimination":
                        text_splited[i][j] = NegationOfNegation()
                    case "TND":
                        text_splited[i][j] = TND()
            else:
                idxs.append(int(text_splited[i][j]))
                if j == len(text_splited[i]) - 1:
                    text_splited[i][3] = idxs
                    text_splited[i] = text_splited[i][:4]
    ans = dict()
    for i in text_splited:
        #print(i)
        if len(i)  == 3:
            ans[i[0]] = structure_of_line_top_down(i[0],[],i[2],ans,formula_text=i[1],BaseContext=base_context)
            #print(to_infix_LP(ans[i[0]].LP),"---")
            if ans[i[0]].LP.conclusion != parse_infix(i[1]):
                raise ValueError("Bad proof")

        else:
            ans[i[0]] = structure_of_line_top_down(i[0],i[3],i[2],ans,formula_text=i[1],BaseContext=base_context)
            #print(to_infix_LP(ans[i[0]].LP),"---")
            if ans[i[0]].LP.conclusion != parse_infix(i[1]):
                raise ValueError("Bad proof")
        #print()
    return ans
proof = """1. s   assumption
2. ¬s   assumption
3. ⊥    ¬-elimination, 2, 1
4. s    RAA, 2–3
5. ⊥    ¬-elimination, 2, 4
6. ¬s    ¬-introduction, 4–3
7. s    RAA, 2–3
8. s    RAA, 2–5
9. ¬¬s    ¬-introduction, 6–3
10. ⊥    ¬-elimination, 9, 6
11. ¬s    ¬-introduction, 7–5
12. s    RAA, 2–5
13. ¬s    ¬-introduction, 12–10
14. ⊥    ¬-elimination, 9, 6
15. ¬¬s    ¬-introduction, 6–14
16. s    ¬¬-elimination, 15
17. ⊥    ¬-elimination, 11, 16
18. ¬s    ¬-introduction, 1–5
19. s    RAA, 11–17
20. ⊥    ¬-elimination, 13, 8
21. ¬¬s    ¬-introduction, 18–20
22. s → ¬¬s    →-introduction, 19–21
23. s    ¬¬-elimination, 21
24. s → ¬¬s    →-introduction, 4–9
25. ¬¬s    →-elimination, 22, 4
26. s    ¬¬-elimination, 15
27. s → ¬¬s    →-introduction, 16–21
28. s    RAA, 13–17
29. s → ⊥    →-introduction, 12–17
30. ¬s    ¬-introduction, 26–20
31. ⊥    →-elimination, 29, 23
32. ⊥    ¬-elimination, 30, 8
33. s    ¬¬-elimination, 25
34. ¬s    ¬-introduction, 29–32"""
base_context = [Negation(parse_infix("s"))]
#parsedProof = parseProof(proof,base_context=base_context)
#for i in parsedProof.keys():



In [26]:
def basing_context_of_LP(x:LittleProblem):
   base_context = x.assumptions.base_context + x.assumptions.additional_context
   return LittleProblem(Context(base_context,[]),x.conclusion)

In [27]:
def evaluate_formula(f,values):
    if isinstance(f,Truth):
        return True
    if isinstance(f,Lie):
        return False
    if isinstance(f,Atom):
        return values[f]
    if isinstance(f,Negation):
        return not evaluate_formula(f.Left(),values)
    if isinstance(f,Conjunction):
        return evaluate_formula(f.Left(),values) and evaluate_formula(f.Right(),values)
    if isinstance(f,Disjunction):
        return evaluate_formula(f.Left(),values) or evaluate_formula(f.Right(),values)
    if isinstance(f,Implication):
        return (not evaluate_formula(f.Left(),values)) or evaluate_formula(f.Right(),values)
    if isinstance(f,Iff):
        return evaluate_formula(f.Left(),values) == evaluate_formula(f.Right(),values)

evaluate_formula(Iff(parse_infix("s"),parse_infix("s")),{parse_infix("s"):True,parse_infix("q"):False})


True

In [28]:
def formula_free_variables(f):
    if not isinstance(f,Formula):
        raise TypeError("Bad arguments")
    if isinstance(f,Lie) or isinstance(f,Truth):
        return []
    if isinstance(f,Atom):
        return [f]
    else:
        ans = []
        for i in f.Interior:
            ans += formula_free_variables(i)
        return list(set(ans))

def free_variables_LP(f):
    if not isinstance(f,LittleProblem):
        raise TypeError("Bad arguments")
    ans = formula_free_variables(f.conclusion)
    for i in f.assumptions.base_context:
        ans += formula_free_variables(i)
    for i in f.assumptions.additional_context:
        ans += formula_free_variables(i)
    return list(set(ans))
#free_variables_LP(parsedProof[25].LP)

In [29]:
[1,2,3,4][1:]

[2, 3, 4]

In [30]:
def all_possibilities(l):
    if len(l) == 0:
        return [dict()]
    else:
        ans = []
        old_dicts = all_possibilities(l[1:])
        for i in old_dicts:
            x1 = i.copy()
            x1[l[0]] = True
            x2 = i.copy()
            x2[l[0]] = False
            ans += [x1,x2]
        return ans
all_possibilities(formula_free_variables(parse_infix("p ∨ q ∨ r ∨ w ∨ z ∨ x ∨ c")))

[{R0(x): True,
  R0(c): True,
  R0(r): True,
  R0(w): True,
  R0(q): True,
  R0(z): True,
  R0(p): True},
 {R0(x): True,
  R0(c): True,
  R0(r): True,
  R0(w): True,
  R0(q): True,
  R0(z): True,
  R0(p): False},
 {R0(x): True,
  R0(c): True,
  R0(r): True,
  R0(w): True,
  R0(q): True,
  R0(z): False,
  R0(p): True},
 {R0(x): True,
  R0(c): True,
  R0(r): True,
  R0(w): True,
  R0(q): True,
  R0(z): False,
  R0(p): False},
 {R0(x): True,
  R0(c): True,
  R0(r): True,
  R0(w): True,
  R0(q): False,
  R0(z): True,
  R0(p): True},
 {R0(x): True,
  R0(c): True,
  R0(r): True,
  R0(w): True,
  R0(q): False,
  R0(z): True,
  R0(p): False},
 {R0(x): True,
  R0(c): True,
  R0(r): True,
  R0(w): True,
  R0(q): False,
  R0(z): False,
  R0(p): True},
 {R0(x): True,
  R0(c): True,
  R0(r): True,
  R0(w): True,
  R0(q): False,
  R0(z): False,
  R0(p): False},
 {R0(x): True,
  R0(c): True,
  R0(r): True,
  R0(w): False,
  R0(q): True,
  R0(z): True,
  R0(p): True},
 {R0(x): True,
  R0(c): True,
  R

In [31]:
def is_LP_tautology(LP):
    possibilities = all_possibilities(free_variables_LP(LP))
    assumptions = LP.assumptions.base_context + LP.assumptions.additional_context
    for i in possibilities:
        interesting = True
        for j in assumptions:
            if not evaluate_formula(j,i):
                interesting = False
                break
        if interesting:
            if not evaluate_formula(LP.conclusion,i):
                return False

    return True


print(is_LP_tautology(LittleProblem(
    Context([],[]),
parse_infix("p"))))

False


In [32]:
class smashed_problem:
    def __init__(self,
                 problem:LittleProblem,
                 subproblems:list[LittleProblem],
                 rule:Rule):
        self.problem = problem
        self.subproblems = subproblems
        self.rule = rule
    def __str__(self):
        ans = to_infix_LP(self.problem) + " ["
        for i in self.subproblems:
            ans += to_infix_LP(i) + " . "
        if ans[-1] != '[':
            ans = ans[:-3]
        ans += "] "
        ans += str(self.rule)
        return ans



def smash_with_proof(line,proof_so_far):
    subproblems = []
    args_keys = []
    match line.rule:
        case Assumption():
            args_keys = []
        case Weakening():
            args_keys = [line.args_keys[0]]
        case ImplicationIntroduction():
            args_keys = [line.args_keys[1]]
        case NegationIntroduction():
            args_keys = [line.args_keys[1]]
        case ImplicationElimination():
            args_keys = [line.args_keys[0],line.args_keys[1]]
        case NegationElimination():
            args_keys = [line.args_keys[0],line.args_keys[1]]
        case ConjunctionIntroduction():
            args_keys = [line.args_keys[0],line.args_keys[1]]
        case DisjunctionIntroduction1():
            args_keys = [line.args_keys[0]]
        case DisjunctionIntroduction2():
            args_keys = [line.args_keys[0]]
        case TruthIntroduction():
            args_keys = []
        case ConjunctionElimination1():
            args_keys = [line.args_keys[0]]
        case ConjunctionElimination2():
            args_keys = [line.args_keys[0]]
        case DisjunctionElimination():
            args_keys = [line.args_keys[0],line.args_keys[1],line.args_keys[2]]
        case LieElimination():
            args_keys = [line.args_keys[0]]
        case IffIntroduction():
            args_keys = [line.args_keys[0],line.args_keys[1]]
        case IffElimination1():
            args_keys = [line.args_keys[0],line.args_keys[1]]
        case IffElimination2():
            args_keys = [line.args_keys[0],line.args_keys[1]]
        case RAA():
            args_keys = [line.args_keys[1]]
        case NegationOfNegation():
            args_keys = [line.args_keys[0]]
        case TND():
            args_keys = []

    for i in args_keys:
        subproblems.append(proof_so_far[i])
    for i in range(subproblems.__len__()):
        subproblems[i] = basing_context_of_LP(subproblems[i].LP)
    problem = line.LP
    rule = line.rule
    return smashed_problem(problem,subproblems,rule)



to_infix(Iff(Lie(),Lie()))


'⊥ ↔ ⊥'

In [33]:
def is_smashed_correctly(SP):
    if not isinstance(SP,smashed_problem):
        raise TypeError("Bad arguments")
    if not isinstance(SP.problem,LittleProblem):
        raise TypeError("Bad arguments")
    if not isinstance(SP.subproblems,list):
        raise TypeError("Bad arguments")
    for i in SP.subproblems:
        if not isinstance(i,LittleProblem):
            raise TypeError("Bad arguments")
        if not i.assumptions.additional_context == []:
            raise TypeError("Bad arguments")
    if not isinstance(SP.rule,Rule):
        raise TypeError("Bad arguments")
    unnormalised_subproblems = []
    def unnormalize_subproblem(x):
        base_context = SP.problem.assumptions.base_context
        additional_context = []
        for i in x.assumptions.base_context:
            if i not in base_context:
                additional_context += [i]
        for i in x.assumptions.additional_context:
            if i not in base_context:
                additional_context += [i]
        conclusion = x.conclusion
        return LittleProblem(Context(base_context,additional_context),conclusion)
    for i in SP.subproblems:
        if not is_LP_tautology(i):
            raise ValueError("Bad arguments")
        unnormalised_subproblems.append(unnormalize_subproblem(i))
    match SP.rule:
        case Assumption():
            phi=SP.problem.conclusion
            BaseContext = SP.problem.assumptions.base_context
            return SP.problem == SP.rule.top_down(unnormalised_subproblems,phi=phi,BaseContext = BaseContext)
        case Weakening():
            psi = SP.problem.conclusion
            return SP.problem == SP.rule.top_down(unnormalised_subproblems,psi=psi)
        case ImplicationIntroduction():
            phi = SP.problem.conclusion.Left()
            return SP.problem == SP.rule.top_down(unnormalised_subproblems,phi=phi)
        case NegationIntroduction():
            phi = SP.problem.conclusion.Left()
            return SP.problem == SP.rule.top_down(unnormalised_subproblems,phi=phi)
        case ImplicationElimination():
            return SP.problem == SP.rule.top_down(unnormalised_subproblems)
        case NegationElimination():
            return SP.problem == SP.rule.top_down(unnormalised_subproblems)
        case ConjunctionIntroduction():
            return SP.problem == SP.rule.top_down(unnormalised_subproblems)
        case DisjunctionIntroduction1():
            psi = SP.problem.conclusion.Right()
            return SP.problem == SP.rule.top_down(unnormalised_subproblems,psi=psi)
        case DisjunctionIntroduction2():
            phi = SP.problem.conclusion.Left()
            return SP.problem == SP.rule.top_down(unnormalised_subproblems,phi = phi)
        case TruthIntroduction():
            BaseContext = SP.problem.assumptions.base_context
            return SP.problem == SP.rule.top_down(unnormalised_subproblems,BaseContext)
        case ConjunctionElimination1():
            return SP.problem == SP.rule.top_down(unnormalised_subproblems)
        case ConjunctionElimination2():
            return SP.problem == SP.rule.top_down(unnormalised_subproblems)
        case DisjunctionElimination():
            return SP.problem == SP.rule.top_down(unnormalised_subproblems)
        case LieElimination():
            phi = SP.problem.conclusion
            return SP.problem == SP.rule.top_down(unnormalised_subproblems,phi=phi)
        case IffIntroduction():
            return SP.problem == SP.rule.top_down(unnormalised_subproblems)
        case IffElimination1():
            return SP.problem == SP.rule.top_down(unnormalised_subproblems)
        case IffElimination2():
            return SP.problem == SP.rule.top_down(unnormalised_subproblems)
        case RAA():
            phi = SP.problem.conclusion
            return SP.problem == SP.rule.top_down(unnormalised_subproblems,phi=phi)
        case NegationOfNegation():
            phi = SP.problem.conclusion
            return SP.problem == SP.rule.top_down(unnormalised_subproblems)
        case TND():
            BaseContext = SP.problem.assumptions.base_context
            phi = SP.problem.conclusion.Left()
            return SP.problem == SP.rule.top_down(unnormalised_subproblems,phi=phi,BaseContext = BaseContext)
            pass


proof = """1. b   assumption
2. y ∨ ¬y    TND"""
parsedProof = parseProof(proof)


for i in parsedProof.keys():
    print(is_smashed_correctly(smash_with_proof(parsedProof[i],parsedProof)))


True
True


In [34]:
"""1. s   assumption
2. ¬s   assumption
3. ⊥    ¬-elimination, 2, 1
4. s    RAA, 2–3
5. ⊥    ¬-elimination, 2, 4
6. ¬s    ¬-introduction, 4–3
7. s    RAA, 2–3
8. s    RAA, 2–5
9. ¬¬s    ¬-introduction, 6–3
10. ⊥    ¬-elimination, 9, 6
11. ¬s    ¬-introduction, 7–5
12. s    RAA, 2–5
13. ¬s    ¬-introduction, 12–10
14. ⊥    ¬-elimination, 9, 6
15. ¬¬s    ¬-introduction, 6–14
16. s    ¬¬-elimination, 15
17. ⊥    ¬-elimination, 11, 16
18. ¬s    ¬-introduction, 1–5
19. s    RAA, 11–17
20. ⊥    ¬-elimination, 13, 8
21. ¬¬s    ¬-introduction, 18–20
22. s → ¬¬s    →-introduction, 19–21
23. ¬¬s    →-elimination, 22, 16
24. s    ¬¬-elimination, 21
25. s → ¬¬s    →-introduction, 4–9
26. ⊤    ⊥-elimination, 20
27. ¬¬s    →-elimination, 22, 4
28. s    ¬¬-elimination, 15
29. s → ¬¬s    →-introduction, 16–21
30. ¬s    ¬-introduction, 4–10
31. s    RAA, 6–14
32. s    RAA, 13–17
33. s → ⊥    →-introduction, 12–17
34. ¬s    ¬-introduction, 32–20
35. s    ¬¬-elimination, 23
36. ⊥    →-elimination, 33, 24
37. ⊤ ∧ s    ∧-introduction, 26, 31
38. ⊥    ¬-elimination, 34, 8
39. s    ¬¬-elimination, 27
40. ¬s    ¬-introduction, 39–38
41. ¬¬s    →-elimination, 25, 28
42. s    RAA, 11–36
43. ⊥    ¬-elimination, 27, 40
44. ¬s    ¬-introduction, 24–38
45. ¬¬s    →-elimination, 29, 42
46. s    ¬¬-elimination, 41
47. ⊥    ¬-elimination, 44, 35
48. s    ∧-elimination2, 37
49. ¬s    ¬-introduction, 46–43
50. ⊥    ¬-elimination, 49, 42
51. s    ¬¬-elimination, 45
52. ¬s    ¬-introduction, 51–50
53. ⊥    →-elimination, 33, 48
54. s    RAA, 30–47
55. ¬¬s    ¬-introduction, 52–53
56. s → ¬¬s    →-introduction, 54–55"""

'1. s   assumption\n2. ¬s   assumption\n3. ⊥    ¬-elimination, 2, 1\n4. s    RAA, 2–3\n5. ⊥    ¬-elimination, 2, 4\n6. ¬s    ¬-introduction, 4–3\n7. s    RAA, 2–3\n8. s    RAA, 2–5\n9. ¬¬s    ¬-introduction, 6–3\n10. ⊥    ¬-elimination, 9, 6\n11. ¬s    ¬-introduction, 7–5\n12. s    RAA, 2–5\n13. ¬s    ¬-introduction, 12–10\n14. ⊥    ¬-elimination, 9, 6\n15. ¬¬s    ¬-introduction, 6–14\n16. s    ¬¬-elimination, 15\n17. ⊥    ¬-elimination, 11, 16\n18. ¬s    ¬-introduction, 1–5\n19. s    RAA, 11–17\n20. ⊥    ¬-elimination, 13, 8\n21. ¬¬s    ¬-introduction, 18–20\n22. s → ¬¬s    →-introduction, 19–21\n23. ¬¬s    →-elimination, 22, 16\n24. s    ¬¬-elimination, 21\n25. s → ¬¬s    →-introduction, 4–9\n26. ⊤    ⊥-elimination, 20\n27. ¬¬s    →-elimination, 22, 4\n28. s    ¬¬-elimination, 15\n29. s → ¬¬s    →-introduction, 16–21\n30. ¬s    ¬-introduction, 4–10\n31. s    RAA, 6–14\n32. s    RAA, 13–17\n33. s → ⊥    →-introduction, 12–17\n34. ¬s    ¬-introduction, 32–20\n35. s    ¬¬-elimination, 2

In [35]:
proof = '''1. w ↔ ¬v   assumption
2. w   assumption
3. ¬v    ↔-elimination, 1, 2
4. w    ↔-elimination, 1, 3
5. w    ↔-elimination, 1, 3
6. (w ↔ ¬v) → w    →-introduction, 1–4
7. w → ((w ↔ ¬v) → w)    →-introduction, 5–6'''


def repairProof(proof):
    text_splited = []
    proof = proof.split("\n")
    for i in proof:
        i_splited = i.split(".")
        i_splited = [i_splited[0]] + i_splited[1].split("  ")
        i_splited = list(filter(lambda x: x!= '',i_splited))
        ii = i_splited[:2]
        for I in i_splited[2:]:
            ii += i_splited[2].split(",")
        i_splited = list(filter(lambda x: x!= '',i_splited))
        i_splited_final = []
        for j in i_splited:
            i_splited_final += j.split("–")
        i_splited = i_splited_final
        i_splited_final = []
        for j in i_splited:
            i_splited_final += j.split(",")
        i_splited_final = list(filter(lambda x: x!= '',i_splited_final))
        for I in range(i_splited_final.__len__()):
            i_splited_final[I] = i_splited_final[I].replace(" ", "")
            i_splited_final[I] = i_splited_final[I].replace(",","")
        text_splited.append(i_splited_final)
    for i in range(len(text_splited)):
        idxs = []
        for j in range(len(text_splited[i])):
            if j == 0:
                text_splited[i][j] = int(text_splited[i][j])
            elif j == 1:
                pass
            elif j == 2:
                text_splited[i][j] = text_splited[i][j].replace(" ", "")
                match text_splited[i][j]:
                    case "∨-introduction":
                        f_conclusion = parse_infix(text_splited[i][1])
                        arg_idx = int(text_splited[i][3])-1
                        f_arg = parse_infix(text_splited[arg_idx][1])
                        if f_conclusion.Left() == f_arg:
                            proof[i] = proof[i].replace("∨-introduction","∨-introduction1")
                        elif f_conclusion.Right() == f_arg:
                            proof[i] = proof[i].replace("∨-introduction","∨-introduction2")
                        else:
                            raise TypeError("Bad proof")
                    case "∧-elimination":
                        f_conclusion = parse_infix(text_splited[i][1])
                        arg_idx = int(text_splited[i][3])-1
                        f_arg = parse_infix(text_splited[arg_idx][1])
                        if f_conclusion == f_arg.Left():
                            proof[i] = proof[i].replace("∧-elimination","∧-elimination1")
                        elif f_conclusion == f_arg.Right():
                            proof[i] = proof[i].replace("∧-elimination","∧-elimination2")
                        else:
                            raise TypeError("Bad proof")
                    case "↔-elimination":
                        f_conclusion = parse_infix(text_splited[i][1])
                        arg_idx = int(text_splited[i][3])-1
                        f_arg = parse_infix(text_splited[arg_idx][1])
                        if f_conclusion == f_arg.Right():
                            proof[i] = proof[i].replace("↔-elimination","↔-elimination1")
                        elif f_conclusion == f_arg.Left():
                            proof[i] = proof[i].replace("↔-elimination","↔-elimination2")
                        else:
                            raise TypeError("Bad proof")
            else:
                idxs.append(int(text_splited[i][j]))
                if j == len(text_splited[i]) - 1:
                    text_splited[i][3] = idxs
                    text_splited[i] = text_splited[i][:4]
    ans = ""
    for i in proof:
        ans+=i
        ans+="\n"
    return ans[:-1]


print(repairProof(proof))

1. w ↔ ¬v   assumption
2. w   assumption
3. ¬v    ↔-elimination1, 1, 2
4. w    ↔-elimination2, 1, 3
5. w    ↔-elimination2, 1, 3
6. (w ↔ ¬v) → w    →-introduction, 1–4
7. w → ((w ↔ ¬v) → w)    →-introduction, 5–6


In [36]:
'''
from tqdm import tqdm

with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()


text = text.splitlines()
text = list(filter(lambda x: x!= '',text))
new_text = ""
for i in enumerate(text):
    if(i[0] % 1000 == 0):
        print(i[0]," / ", len(text))
    if i[1][0] == "P" and i[0] != 0:
        new_text += "\n"
    new_text += i[1]
    new_text += "\n"

text = new_text
text = [p.strip() for p in re.split(r'\n\s*\n', text.strip()) if p.strip()]
for i in enumerate(text):
    first, *rest = i[1].split("\n", 1)
    #print(first,i[0])
    first = first[12:]
    rest = rest[0] if rest else ""
    text[i[0]] = [first, rest]

for i in text:
    print(i[0])
'''


<>:1: SyntaxWarning: invalid escape sequence '\s'
<>:1: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_75899/1001326670.py:1: SyntaxWarning: invalid escape sequence '\s'
  '''


'\nfrom tqdm import tqdm\n\nwith open("input.txt", "r", encoding="utf-8") as f:\n    text = f.read()\n\n\ntext = text.splitlines()\ntext = list(filter(lambda x: x!= \'\',text))\nnew_text = ""\nfor i in enumerate(text):\n    if(i[0] % 1000 == 0):\n        print(i[0]," / ", len(text))\n    if i[1][0] == "P" and i[0] != 0:\n        new_text += "\n"\n    new_text += i[1]\n    new_text += "\n"\n\ntext = new_text\ntext = [p.strip() for p in re.split(r\'\n\\s*\n\', text.strip()) if p.strip()]\nfor i in enumerate(text):\n    first, *rest = i[1].split("\n", 1)\n    #print(first,i[0])\n    first = first[12:]\n    rest = rest[0] if rest else ""\n    text[i[0]] = [first, rest]\n\nfor i in text:\n    print(i[0])\n'

In [37]:
'''
import re
text = new_text
text = [p.strip() for p in re.split(r'\n\s*\n', text.strip()) if p.strip()]
for i in enumerate(text):
    first, *rest = i[1].split("\n", 1)
    #print(first,i[0])
    first = first[12:]
    rest = rest[0] if rest else ""
    text[i[0]] = [first, rest]

for i in text:
    print(i[0])
'''

<>:1: SyntaxWarning: invalid escape sequence '\s'
<>:1: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_75899/2740104964.py:1: SyntaxWarning: invalid escape sequence '\s'
  '''


'\nimport re\ntext = new_text\ntext = [p.strip() for p in re.split(r\'\n\\s*\n\', text.strip()) if p.strip()]\nfor i in enumerate(text):\n    first, *rest = i[1].split("\n", 1)\n    #print(first,i[0])\n    first = first[12:]\n    rest = rest[0] if rest else ""\n    text[i[0]] = [first, rest]\n\nfor i in text:\n    print(i[0])\n'

In [38]:
'''
proof = text[1]


def replace_variable_in_formula(f:Formula,x:str,y:str):
    if isinstance(f,Atom):
        if f == parse_infix(x):
            return parse_infix(y)
        else:
            return f
    elif isinstance(f,Truth) or isinstance(f,Lie):
        return f
    else:
        for i in range(len(f.Interior)):
            f.Interior[i] = replace_variable_in_formula(f.Interior[i],x,y)
    return f

def replace_variable_in_proof_splitted_and_repair(p,x:str,y:str):
    ans_splited = [to_infix(replace_variable_in_formula(parse_infix(p[0]),x,y))+"\n"]
    proof = p[1].splitlines()
    for i in range(len(proof)):
        proof[i] = proof[i].split(".")
        proof[i] = [proof[i][0]] + proof[i][1].split("  ")
        proof[i] = [j for j in proof[i] if j != ""]
        proof[i][1] = to_infix(replace_variable_in_formula(parse_infix(proof[i][1]),x,y))
        if len(proof[i]) == 2:
            ans_splited.append(proof[i][0]+". "+proof[i][1]+"\n")
        else:
            ans_splited.append(proof[i][0]+". "+proof[i][1]+"   "+proof[i][2]+"\n")
    proof_without_goal = ""
    for i in ans_splited[1:]:
        proof_without_goal += i
    proof_without_goal = repairProof(proof_without_goal[:-1])
    return "Prove that: " + ans_splited[0] + proof_without_goal


for i in enumerate(text):
    if i[0]  % 100 == 0:
        print(i[0]," / ",len(text))
    free_vars = formula_free_variables(parse_infix(i[1][0]))
    for j in range(len(free_vars)):
        free_vars[j] = to_infix(free_vars[j])
    vars_to_replace = []
    for j in range(len(free_vars)):
        while True:
            candidate = random.choice(list("abcdefghijklmnoqprstuvwxyz"))
            if not ((candidate in vars_to_replace) or (candidate in free_vars)):
                vars_to_replace.append(candidate)
                break
    #for j in i:
    #    print(j)
    new_proof = i[1]
    #print(new_proof)
    for j in range(len(free_vars)):
        x = new_proof
        new_proof = replace_variable_in_proof_splitted_and_repair(x,free_vars[j],vars_to_replace[j])

        first, *rest = new_proof.split("\n", 1)
        first = first[12:]
        rest = rest[0] if rest else ""
        new_proof = [first, rest]
        text[i[0]] = new_proof
    #print(vars_to_replace)
'''

'\nproof = text[1]\n\n\ndef replace_variable_in_formula(f:Formula,x:str,y:str):\n    if isinstance(f,Atom):\n        if f == parse_infix(x):\n            return parse_infix(y)\n        else:\n            return f\n    elif isinstance(f,Truth) or isinstance(f,Lie):\n        return f\n    else:\n        for i in range(len(f.Interior)):\n            f.Interior[i] = replace_variable_in_formula(f.Interior[i],x,y)\n    return f\n\ndef replace_variable_in_proof_splitted_and_repair(p,x:str,y:str):\n    ans_splited = [to_infix(replace_variable_in_formula(parse_infix(p[0]),x,y))+"\n"]\n    proof = p[1].splitlines()\n    for i in range(len(proof)):\n        proof[i] = proof[i].split(".")\n        proof[i] = [proof[i][0]] + proof[i][1].split("  ")\n        proof[i] = [j for j in proof[i] if j != ""]\n        proof[i][1] = to_infix(replace_variable_in_formula(parse_infix(proof[i][1]),x,y))\n        if len(proof[i]) == 2:\n            ans_splited.append(proof[i][0]+". "+proof[i][1]+"\n")\n        el

In [39]:
'''
text_v2 = ""
for i in text:
    text_v2 += "Prove that: "
    text_v2 += i[0]
    text_v2 += "\n"
    text_v2 += i[1]
    text_v2 += "\n\n"
'''

'\ntext_v2 = ""\nfor i in text:\n    text_v2 += "Prove that: "\n    text_v2 += i[0]\n    text_v2 += "\n"\n    text_v2 += i[1]\n    text_v2 += "\n\n"\n'

In [40]:
'''
with open("inputv2.txt", "w", encoding="utf-8") as f:
    f.write(text_v2)
'''

'\nwith open("inputv2.txt", "w", encoding="utf-8") as f:\n    f.write(text_v2)\n'

In [41]:
import re
with open("inputv2.txt", "r", encoding="utf-8") as f:
    text = f.read()

text = [p.strip() for p in re.split(r'\n\s*\n', text.strip()) if p.strip()]
for i in enumerate(text):
    first, *rest = i[1].split("\n", 1)
    #print(first,i[0])
    first = first[12:]
    rest = rest[0] if rest else ""
    text[i[0]] = [first, rest]

In [42]:
len(text)

23410

In [43]:
'''
from tqdm import tqdm
for i in tqdm(range(len(text))):
    parsedProof = parseProof(text[i][1])
    for j in parsedProof.keys():
        SP = smash_with_proof(parsedProof[j],parsedProof)
        if not is_smashed_correctly(SP):
            raise TypeError("Bad smashed")
'''


'\nfrom tqdm import tqdm\nfor i in tqdm(range(len(text))):\n    parsedProof = parseProof(text[i][1])\n    for j in parsedProof.keys():\n        SP = smash_with_proof(parsedProof[j],parsedProof)\n        if not is_smashed_correctly(SP):\n            raise TypeError("Bad smashed")\n'

In [44]:
parsed_proof1 = parseProof(text[0][1])
for i in parsed_proof1.keys():
    print(parsed_proof1[i])
    print(smash_with_proof(parsed_proof1[i],parsed_proof1))
    print()

print(parsed_proof1)

1 []  ; s ∨ c ⊢ s ∨ c Assumption
 ; s ∨ c ⊢ s ∨ c [] Assumption

2 [1]  ; s ∨ c, s ⊢ s Weakening
 ; s ∨ c, s ⊢ s [s ∨ c  ⊢ s ∨ c] Weakening

3 [2]  ; s ∨ c, s ⊢ c ∨ s DisjunctionIntroduction2
 ; s ∨ c, s ⊢ c ∨ s [s ∨ c, s  ⊢ s] DisjunctionIntroduction2

4 [1]  ; c, s ∨ c ⊢ c Weakening
 ; c, s ∨ c ⊢ c [s ∨ c  ⊢ s ∨ c] Weakening

5 [4]  ; c, s ∨ c ⊢ c ∨ s DisjunctionIntroduction1
 ; c, s ∨ c ⊢ c ∨ s [c, s ∨ c  ⊢ c] DisjunctionIntroduction1

6 [1, 3, 5]  ; s ∨ c ⊢ c ∨ s DisjunctionElimination
 ; s ∨ c ⊢ c ∨ s [s ∨ c  ⊢ s ∨ c . s ∨ c, s  ⊢ c ∨ s . c, s ∨ c  ⊢ c ∨ s] DisjunctionElimination

7 [1, 6]   ⊢ (s ∨ c) → (c ∨ s) ImplicationIntroduction
  ⊢ (s ∨ c) → (c ∨ s) [s ∨ c  ⊢ c ∨ s] ImplicationIntroduction

{1: 1 []  ; R0(s) ∨ R0(c) ⊢ R0(s) ∨ R0(c) Assumption, 2: 2 [1]  ; R0(s) ∨ R0(c), R0(s) ⊢ R0(s) Weakening, 3: 3 [2]  ; R0(s) ∨ R0(c), R0(s) ⊢ R0(c) ∨ R0(s) DisjunctionIntroduction2, 4: 4 [1]  ; R0(c), R0(s) ∨ R0(c) ⊢ R0(c) Weakening, 5: 5 [4]  ; R0(c), R0(s) ∨ R0(c) ⊢ R0(c) ∨ R0(s) Disjun

In [45]:
proof_text = text[0]

SP = smash_with_proof(parseProof(proof_text[1])[1],parseProof(proof_text[1]))

In [46]:
proof = repairProof(proof)

In [47]:
import re
def almost_equal(x:LittleProblem,y:LittleProblem):
    return basing_context_of_LP(x) == basing_context_of_LP(y)






def extract_proof(proof:str,LP:LittleProblem,BaseContext = []):
    '''
    BaseContext to kontekst bazowy w którym dzieje się proof
    '''
    parsedProof = parseProof(proof,base_context=BaseContext)
    goal_idx = -1
    for i in parsedProof.keys():
        if almost_equal(parsedProof[i].LP,LP):
            goal_idx = i
            break
    if goal_idx == -1:
        raise TypeError("Cannot extract")
    important_lines = [goal_idx]
    def next_elems():

        ans = []
        for i in parsedProof.keys():
            if not i in important_lines:
                for j in  important_lines:
                    if i in parsedProof[j].args_keys:
                        ans.append(i)
                        break

        return ans
    next_elems_values = next_elems()
    while next_elems_values != []:
        important_lines += next_elems_values
        next_elems_values = next_elems()
    important_lines.sort()
    ans = ""
    proof = proof.splitlines()
    for i in proof:
        idx = int(i.split(".")[0])
        if idx in important_lines:
            ans += i
            ans += "\n"
    ans = ans[:-1]
    return ans
    def replace_indexed_numbers(ls: List[str], li: List[int]) -> List[str]:
        mapping = {int(li[i]): str(i+1) for i in range(len(li))}
        if not mapping:
            return ls[:]

        alts = "|".join(
            re.escape(str(n)) for n in sorted(mapping.keys(), key=lambda x: (-len(str(x)), x))
        )

        pattern = re.compile(
            rf'(?:(, |–)({alts})(?!\d))|(?:(?<!\d)({alts})(?!\d)(?=\.))'
        )

        def repl(m: re.Match) -> str:
            if m.group(2) is not None:  # gałąź z prefiksem
                prefix = m.group(1)
                num = m.group(2)
                return prefix + mapping[int(num)]
            else:  # gałąź z kropką po liczbie
                num = m.group(3)
                return mapping[int(num)]

        ans_lines = [pattern.sub(repl, s) for s in ls]
        ans = "\n".join(ans_lines)
        return ans

    return replace_indexed_numbers(ans.splitlines(),important_lines)


proof = '''1. c ∨ k    assumption
2. k   weakening, 1
3. (c ∨ k) → k    →-introduction, 1–2
4. k → ((c ∨ k) → k)    →-introduction, 2–3'''
parseProof(proof)

LP = LittleProblem(Context([],[]),parse_infix("k → ((c ∨ k) → k)"))

print(extract_proof(proof,LP))


print(LP)
#print(proof)

print(proof)
base_context = [parse_infix("w")]

def simpliffy_with_context(proof,context):
    def find_line_idx(idx, lines):
        for i in range(len(lines)):
            if lines[i].split(".")[0] == str(idx):
                return i

    parsedProof = parseProof(proof,context)
    proof_text = proof.splitlines()
    for i in parsedProof.keys():
        if parsedProof[i].LP.conclusion in context:
            line_conclusion_idx = find_line_idx(i,proof_text)
            line_assumption_idx = -1
            for j in parsedProof[i].args_keys:
                if int(i) > int(j) and parsedProof[i].LP.assumptions == parsedProof[j].LP.assumptions:
                    line_assumption_idx = j
                    break
            if line_assumption_idx != -1:
                new_line = str(i)+". "+ to_infix(parsedProof[i].LP.conclusion)+"   weakening, "+str(line_assumption_idx)
                proof_text[line_conclusion_idx] = new_line
    ans = ""
    for i in proof_text:
        ans += i
        ans += "\n"
    ans = ans[:-1]






    parsedProof = parseProof(ans,base_context = context)



    important_lines = [list(parsedProof.keys())[-1]]
    def next_elems():

        ans = []
        for i in parsedProof.keys():
            if not i in important_lines:
                for j in  important_lines:
                    if i in parsedProof[j].args_keys:
                        ans.append(i)
                        break

        return ans
    next_elems_values = next_elems()
    while next_elems_values != []:
        important_lines += next_elems_values
        next_elems_values = next_elems()
    important_lines.sort()
    proof = ans.splitlines()
    ans = ""

    for i in proof:
        idx = int(i.split(".")[0])
        if idx in important_lines:
            ans += i
            ans += "\n"
    ans = ans[:-1]

    def replace_indexed_numbers(ls: List[str], li: List[int]) -> List[str]:
        mapping = {int(li[i]): str(i+1) for i in range(len(li))}
        if not mapping:
            return ls[:]

        # najdłuższe liczby najpierw, żeby nie łapać 20 w 200
        alts = "|".join(
            re.escape(str(n)) for n in sorted(mapping.keys(), key=lambda x: (-len(str(x)), x))
        )

        # 1) prefiks ", " albo "–" + liczba (nie jako fragment większej)
        # 2) liczba (nie jako fragment większej) + kropka po niej
        pattern = re.compile(
            rf'(?:(, |–)({alts})(?!\d))|(?:(?<!\d)({alts})(?!\d)(?=\.))'
        )

        def repl(m: re.Match) -> str:
            if m.group(2) is not None:  # gałąź z prefiksem
                prefix = m.group(1)
                num = m.group(2)
                return prefix + mapping[int(num)]
            else:  # gałąź z kropką po liczbie
                num = m.group(3)
                return mapping[int(num)]

        ans_lines = [pattern.sub(repl, s) for s in ls]
        ans = "\n".join(ans_lines)
        return ans

    return replace_indexed_numbers(ans.splitlines(),important_lines)






print()

proof2 = simpliffy_with_context(proof,base_context)
print()
print()
print()
print()
print(proof2)

parsedProof2 = parseProof(proof2)
print(parsedProof2)

1. c ∨ k    assumption
2. k   weakening, 1
3. (c ∨ k) → k    →-introduction, 1–2
4. k → ((c ∨ k) → k)    →-introduction, 2–3
 ;   ⊢ '(k) → (('(c) ∨ '(k)) → '(k))
1. c ∨ k    assumption
2. k   weakening, 1
3. (c ∨ k) → k    →-introduction, 1–2
4. k → ((c ∨ k) → k)    →-introduction, 2–3





1. c ∨ k    assumption
2. k   weakening, 1
3. (c ∨ k) → k    →-introduction, 1–2
4. k → ((c ∨ k) → k)    →-introduction, 2–3
{1: 1 []  ; R0(c) ∨ R0(k) ⊢ R0(c) ∨ R0(k) Assumption, 2: 2 [1]  ; R0(k), R0(c) ∨ R0(k) ⊢ R0(k) Weakening, 3: 3 [1, 2]  ; R0(k) ⊢ (R0(c) ∨ R0(k)) → R0(k) ImplicationIntroduction, 4: 4 [2, 3]  ;   ⊢ R0(k) → ((R0(c) ∨ R0(k)) → R0(k)) ImplicationIntroduction}


In [48]:
from copy import deepcopy
def check_all_extraction(proof:str,BaseContext = []):
    parsedProof = parseProof(proof,base_context = BaseContext)
    for i in parsedProof.keys():
        goal = parsedProof[i].LP
        temporary_context = deepcopy(BaseContext)
        for j in goal.assumptions.base_context:
            if not j in BaseContext:
                temporary_context.append(j)
        for j in goal.assumptions.additional_context:
            if not j in BaseContext:
                temporary_context.append(j)
        goal.assumptions.base_context = temporary_context#?
        goal.assumptions.additional_context = []#?
        extracted = extract_proof(proof,goal,temporary_context)
        parseProof(extracted,temporary_context)
        extracted = simpliffy_with_context(extracted,temporary_context)
        parseProof(extracted,base_context = temporary_context)

'''

for i in tqdm(range(len(text))):
    check_all_extraction(text[i][1])
'''

'\n\nfor i in tqdm(range(len(text))):\n    check_all_extraction(text[i][1])\n'

In [49]:
def give_all_extraction_text(proof:str,BaseContext = []):
    parsedProof = parseProof(proof,base_context = BaseContext)
    ANS = ""
    for i in parsedProof.keys():
        goal = parsedProof[i].LP
        temporary_context = deepcopy(BaseContext)
        for j in goal.assumptions.base_context:
            if not j in BaseContext:
                temporary_context.append(j)
        for j in goal.assumptions.additional_context:
            if not j in BaseContext:
                temporary_context.append(j)
        goal.assumptions.base_context = temporary_context#?
        goal.assumptions.additional_context = []#?
        extracted = extract_proof(proof,goal,temporary_context)
        parseProof(extracted,temporary_context)
        extracted = simpliffy_with_context(extracted,temporary_context)
        parseProof(extracted,base_context = temporary_context)
        ans = "Prove that: " + to_infix(goal.conclusion)
        if temporary_context != []:
            ans += "\nassuming that: "
            for j in temporary_context:
                ans += to_infix(j)
                ans += ",\n"
            ans = ans[:-2]
        ans += ("\n"+extracted)
        ANS += ans
        ANS += "\n\n"
    return ANS
give_all_extraction_text(proof)

'Prove that: c ∨ k\nassuming that: c ∨ k\n1. c ∨ k    assumption\n\nProve that: k\nassuming that: k,\nc ∨ k\n1. c ∨ k    assumption\n2. k   weakening, 1\n\nProve that: (c ∨ k) → k\nassuming that: k\n1. c ∨ k    assumption\n2. k   weakening, 1\n3. (c ∨ k) → k    →-introduction, 1–2\n\nProve that: k → ((c ∨ k) → k)\n1. c ∨ k    assumption\n2. k   weakening, 1\n3. (c ∨ k) → k    →-introduction, 1–2\n4. k → ((c ∨ k) → k)    →-introduction, 2–3\n\n'

In [50]:
'''
NEW_CORPUS_PROVING = ""

for i in tqdm(range(len(text))):
    NEW_CORPUS_PROVING += give_all_extraction_text(text[i][1])
'''

'\nNEW_CORPUS_PROVING = ""\n\nfor i in tqdm(range(len(text))):\n    NEW_CORPUS_PROVING += give_all_extraction_text(text[i][1])\n'

Kolejny pomysł: Sensitivity

In [51]:
'''
with open("NEW_CORPUS_PROVING.txt", "w", encoding="utf-8") as f:
    f.write(NEW_CORPUS_PROVING)
'''

'\nwith open("NEW_CORPUS_PROVING.txt", "w", encoding="utf-8") as f:\n    f.write(NEW_CORPUS_PROVING)\n'

In [52]:
'''
for i in NEW_CORPUS_PROVING.splitlines():
    if i[:2]== "1.":
        print(i)
'''

'\nfor i in NEW_CORPUS_PROVING.splitlines():\n    if i[:2]== "1.":\n        print(i)\n'

In [53]:
'''
with open("NEW_CORPUS_PROVING.txt", "r", encoding="utf-8") as f:
    text = f.read()

text = text.rstrip("\n")
'''

'\nwith open("NEW_CORPUS_PROVING.txt", "r", encoding="utf-8") as f:\n    text = f.read()\n\ntext = text.rstrip("\n")\n'

In [54]:
'''
text = text.split("\n\n")
'''

'\ntext = text.split("\n\n")\n'

In [55]:
def check_proof_with_assumptions(proof):
    proof = proof.splitlines()
    thesis_conclusion = parse_infix(proof[0][12:])
    base_context = []
    if proof[1][0] == "a":
        if proof[1][-1] == ",":
            base_context.append(parse_infix(proof[1][15:-1]))
            j = 2
            while proof[j][0] != "1":
                if proof[j][-1] == ",":
                    base_context.append(parse_infix(proof[j][:-1]))
                else:
                    base_context.append(parse_infix(proof[j]))
                j += 1
        else:
            base_context.append(parse_infix(proof[1][15:]))
    proof = "\n".join(proof[1+len(base_context):])
    parsedProof = parseProof(proof,base_context)
    if parsedProof[len(parsedProof)].LP != LittleProblem(Context(base_context,[]) , thesis_conclusion):
        raise TypeError("Bad proof")
    return parsedProof



In [56]:
'''
with open("inputv2.txt", "r", encoding="utf-8") as f:
    text = f.read()


text = text.splitlines()
text = list(filter(lambda x: x!= '',text))
new_text = ""
for i in enumerate(text):
    if(i[0] % 1000 == 0):
        print(i[0]," / ", len(text))
    if i[1][0] == "P" and i[0] != 0:
        new_text += "\n"
    new_text += i[1]
    new_text += "\n"

text = new_text
text = [p.strip() for p in re.split(r'\n\s*\n', text.strip()) if p.strip()]
for i in enumerate(text):
    first, *rest = i[1].split("\n", 1)
    #print(first,i[0])
    first = first[12:]
    rest = rest[0] if rest else ""
    text[i[0]] = [first, rest]

for i in text:
    print(i[0])
'''

<>:1: SyntaxWarning: invalid escape sequence '\s'
<>:1: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_75899/1499411579.py:1: SyntaxWarning: invalid escape sequence '\s'
  '''


'\nwith open("inputv2.txt", "r", encoding="utf-8") as f:\n    text = f.read()\n\n\ntext = text.splitlines()\ntext = list(filter(lambda x: x!= \'\',text))\nnew_text = ""\nfor i in enumerate(text):\n    if(i[0] % 1000 == 0):\n        print(i[0]," / ", len(text))\n    if i[1][0] == "P" and i[0] != 0:\n        new_text += "\n"\n    new_text += i[1]\n    new_text += "\n"\n\ntext = new_text\ntext = [p.strip() for p in re.split(r\'\n\\s*\n\', text.strip()) if p.strip()]\nfor i in enumerate(text):\n    first, *rest = i[1].split("\n", 1)\n    #print(first,i[0])\n    first = first[12:]\n    rest = rest[0] if rest else ""\n    text[i[0]] = [first, rest]\n\nfor i in text:\n    print(i[0])\n'

In [57]:
def to_infix_LP_simple(LP: LittleProblem) -> str:
    ans = ""
    for i in LP.assumptions.base_context:
        ans += to_infix(i) + ", "
    for i in LP.assumptions.additional_context:
        ans += to_infix(i) + ", "
    if ans != "":
        ans = ans[:-2]
    ans += " ⊢ " + to_infix(LP.conclusion)

    return ans

def pretty_print_smashed_problem(x:smashed_problem):
    ans = "Smash: "
    ans += to_infix_LP_simple(x.problem)
    ans += "\nWith: "
    ans += x.rule.__str__()
    ans += "\nTo: "
    for i in x.subproblems:
        ans += to_infix_LP_simple(i)
        ans += ",\n"
    if x.subproblems != []:
        ans = ans[:-2]
    return ans






In [58]:
'''
NEW_CORPUS_SMASH = ""
for i in tqdm(range(len(text))):
    proof = parseProof(text[i][1])
    for j in proof.keys():
        NEW_CORPUS_SMASH += pretty_print_smashed_problem(smash_with_proof(proof[j], proof))
        NEW_CORPUS_SMASH += "\n\n"
'''

'\nNEW_CORPUS_SMASH = ""\nfor i in tqdm(range(len(text))):\n    proof = parseProof(text[i][1])\n    for j in proof.keys():\n        NEW_CORPUS_SMASH += pretty_print_smashed_problem(smash_with_proof(proof[j], proof))\n        NEW_CORPUS_SMASH += "\n\n"\n'

In [59]:
'''
with open("NEW_CORPUS_SMASH.txt", "w", encoding="utf-8") as f:
    f.write(NEW_CORPUS_SMASH)
'''

'\nwith open("NEW_CORPUS_SMASH.txt", "w", encoding="utf-8") as f:\n    f.write(NEW_CORPUS_SMASH)\n'

In [60]:
'''
with open("NEW_CORPUS_SMASH.txt", "r", encoding="utf-8") as f:
    text = f.read()
text = text.split("\n\n")
text = text[:-1]
'''

'\nwith open("NEW_CORPUS_SMASH.txt", "r", encoding="utf-8") as f:\n    text = f.read()\ntext = text.split("\n\n")\ntext = text[:-1]\n'

In [61]:
def check_smash_from_text(smash):
    smash = smash.splitlines()
    main_problem_unparsed = smash[0][7:]
    main_problem_unparsed = main_problem_unparsed.lstrip(" ")
    main_problem_unparsed = main_problem_unparsed.split("⊢")
    main_problem_unparsed[0] = main_problem_unparsed[0].split(",")
    if main_problem_unparsed[0] == [""]:
        main_problem_unparsed[0] = []
    base_context = []
    for i in main_problem_unparsed[0]:
        base_context.append(parse_infix(i))
    main_conclusion = parse_infix(main_problem_unparsed[1])
    goal = LittleProblem(Context(base_context,[]),conclusion=main_conclusion)
    rule_unparsed = smash[1][6:]
    rule_unparsed = rule_unparsed.lstrip(" ")
    rule_unparsed = rule_unparsed.rstrip(" ")
    rule = None
    match rule_unparsed:
        case 'Assumption':
            rule = Assumption()
        case "Weakening":
            rule = Weakening()
        case "ImplicationIntroduction":
            rule = ImplicationIntroduction()
        case "NegationIntroduction":
            rule = NegationIntroduction()
        case "ImplicationElimination":
            rule = ImplicationElimination()
        case "NegationElimination":
            rule = NegationElimination()
        case "ConjunctionIntroduction":
            rule = ConjunctionIntroduction()
        case "DisjunctionIntroduction1":
            rule = DisjunctionIntroduction1()
        case "DisjunctionIntroduction2":
            rule = DisjunctionIntroduction2()
        case "TruthIntroduction":
            rule = TruthIntroduction()
        case "ConjunctionElimination1":
            rule = ConjunctionElimination1()
        case "ConjunctionElimination2":
            rule = ConjunctionElimination2()
        case "DisjunctionElimination":
            rule = DisjunctionElimination()
        case "LieElimination":
            rule = LieElimination()
        case "IffIntroduction":
            rule = IffIntroduction()
        case "IffElimination1":
            rule = IffElimination1()
        case "IffElimination2":
            rule = IffElimination2()
        case "RAA":
            rule = RAA()
        case "NegationOfNegation":
            rule = NegationOfNegation()
        case "TND":
            rule = TND()
    if rule == None:
        raise TypeError("Bad rule")
    smash[2] = smash[2].rstrip(" ")
    smash[2]= smash[2].rstrip(",")
    for i in range(3,len(smash)):
        smash[i] = smash[i].rstrip(" ")
        smash[i] = smash[i].rstrip(",")


    subproblems= []
    if not smash[2] == "To:":
        subproblems.append(smash[2][3:])
        for i in range(3,len(smash)):
            subproblems.append(smash[i])


    for i in range(len(subproblems)):
        subproblems[i] = subproblems[i].split("⊢")
        subproblems[i][0] = subproblems[i][0].split(",")
        subproblems[i][1] = parse_infix(subproblems[i][1])
        for j in range(len(subproblems[i][0])):
            subproblems[i][0][j] = subproblems[i][0][j].rstrip(" ")
        if subproblems[i][0] == [""]:
            subproblems[i][0] = []
        for j in range(len(subproblems[i][0])):
            subproblems[i][0][j] = parse_infix(subproblems[i][0][j])
        subproblems[i] = LittleProblem(Context(subproblems[i][0],[]),conclusion=subproblems[i][1])
    SP = smashed_problem(goal,subproblems,rule)
    if not is_smashed_correctly(SP):
        raise TypeError("Bad smashed problem")
    return SP

proof = '''Smash:  ⊢ ¬(l → h) ∨ (y → ¬¬y)
With: DisjunctionIntroduction2
To:  ⊢ y → ¬¬y'''


In [62]:
for i in range(3,10):
    print(i)

3
4
5
6
7
8
9


In [63]:



import pickle
import re
import shutil
import subprocess
import sys
from pathlib import Path

import numpy as np


def train_nanogpt_on_file(
    file_name: str,
    dataset_name: str | None = None,
    out_dir: str | None = None,
    train_split: float = 0.9,
    max_iters: int = 20000,
    eval_interval: int = 1000,
    log_interval: int = 1,
    batch_size: int = 12,
    block_size: int = 1024,
    n_layer: int = 6,
    n_head: int = 6,
    n_embd: int = 384,
    learning_rate: float = 2e-2,
    compile_model: bool = False,
    device: str | None = None,
):
    project_root = Path(globals().get('PROJECT_ROOT', Path.cwd()))
    pkg_root = Path(globals().get('PKG_ROOT', project_root / 'nanoGPT-20251228T135841Z-3-001' / 'nanoGPT'))

    if not (pkg_root / 'train.py').exists():
        alt_pkg_root = project_root / 'nanoGPT'
        if (alt_pkg_root / 'train.py').exists():
            pkg_root = alt_pkg_root
        else:
            raise FileNotFoundError(f'Nie znaleziono train.py w {pkg_root} ani {alt_pkg_root}.')

    src = Path(file_name).expanduser()
    if not src.is_absolute():
        from_project = (project_root / src).resolve()
        from_cwd = (Path.cwd() / src).resolve()
        src = from_project if from_project.exists() else from_cwd

    if not src.exists():
        raise FileNotFoundError(f'Nie znaleziono pliku wejsciowego: {src}')

    if dataset_name is None:
        dataset_name = re.sub(r'[^0-9A-Za-z_]+', '_', src.stem).strip('_') or 'custom_char'

    data_dir = pkg_root / 'data' / dataset_name
    data_dir.mkdir(parents=True, exist_ok=True)
    input_txt = data_dir / 'input.txt'
    shutil.copy2(src, input_txt)

    text = input_txt.read_text(encoding='utf-8')
    if len(text) < 2:
        raise ValueError('Plik wejsciowy jest zbyt krotki do podzialu train/val.')

    chars = sorted(set(text))
    vocab_size = len(chars)
    if vocab_size > 65535:
        raise ValueError('Za duzo unikalnych znakow dla kodowania uint16.')

    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for i, ch in enumerate(chars)}

    ids = np.array([stoi[ch] for ch in text], dtype=np.uint16)
    split_idx = int(len(ids) * train_split)
    split_idx = max(1, min(len(ids) - 1, split_idx))

    train_ids = ids[:split_idx]
    val_ids = ids[split_idx:]

    train_ids.tofile(data_dir / 'train.bin')
    val_ids.tofile(data_dir / 'val.bin')

    meta = {
        'vocab_size': vocab_size,
        'itos': itos,
        'stoi': stoi,
    }
    with open(data_dir / 'meta.pkl', 'wb') as f:
        pickle.dump(meta, f)

    if out_dir is None:
        model_out_dir = pkg_root / f'out-{dataset_name}'
    else:
        model_out_dir = Path(out_dir)
        if not model_out_dir.is_absolute():
            model_out_dir = pkg_root / model_out_dir

    if device is None:
        device = 'cuda' if shutil.which('nvidia-smi') else 'cpu'

    print(f'Plik wejsciowy: {src.resolve()}')
    print(f'Model bedzie zapisany w: {model_out_dir.resolve()}')
    print(f'Dataset nanoGPT: {dataset_name}')

    cmd = [
        sys.executable,
        'train.py',
        f'--dataset={dataset_name}',
        f'--out_dir={model_out_dir}',
        f'--max_iters={max_iters}',
        f'--eval_interval={eval_interval}',
        f'--log_interval={log_interval}',
        f'--batch_size={batch_size}',
        f'--block_size={block_size}',
        f'--n_layer={n_layer}',
        f'--n_head={n_head}',
        f'--n_embd={n_embd}',
        f'--learning_rate={learning_rate}',
        f'--device={device}',
        f'--compile={compile_model}',
        '--always_save_checkpoint=True',
    ]

    process = subprocess.Popen(
        cmd,
        cwd=str(pkg_root),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    if process.stdout is None:
        raise RuntimeError('Nie udalo sie przechwycic logow treningu.')

    for line in process.stdout:
        print(line, end='')

    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f'Trening nanoGPT zakonczyl sie kodem {return_code}.')

    return model_out_dir


def finetune_nanogpt_on_file(
    file_name: str,
    base_model: str,
    dataset_name: str | None = None,
    out_dir: str | None = None,
    train_split: float = 0.9,
    max_iters: int = 20000,
    eval_interval: int = 1000,
    log_interval: int = 1,
    batch_size: int = 12,
    block_size: int = 1024,
    learning_rate: float = 1e-3,
    compile_model: bool = False,
    device: str | None = None,
):
    project_root = Path(globals().get('PROJECT_ROOT', Path.cwd()))
    pkg_root = Path(globals().get('PKG_ROOT', project_root / 'nanoGPT-20251228T135841Z-3-001' / 'nanoGPT'))

    if not (pkg_root / 'train.py').exists():
        alt_pkg_root = project_root / 'nanoGPT'
        if (alt_pkg_root / 'train.py').exists():
            pkg_root = alt_pkg_root
        else:
            raise FileNotFoundError(f'Nie znaleziono train.py w {pkg_root} ani {alt_pkg_root}.')

    src = Path(file_name).expanduser()
    if not src.is_absolute():
        from_project = (project_root / src).resolve()
        from_cwd = (Path.cwd() / src).resolve()
        src = from_project if from_project.exists() else from_cwd
    if not src.exists():
        raise FileNotFoundError(f'Nie znaleziono pliku wejsciowego: {src}')

    base_model_path = Path(base_model).expanduser()
    if not base_model_path.is_absolute():
        from_project = (project_root / base_model_path).resolve()
        from_cwd = (Path.cwd() / base_model_path).resolve()
        from_pkg = (pkg_root / base_model_path).resolve()
        if from_project.exists():
            base_model_path = from_project
        elif from_cwd.exists():
            base_model_path = from_cwd
        else:
            base_model_path = from_pkg

    ckpt_src = base_model_path / 'ckpt.pt' if base_model_path.is_dir() else base_model_path
    if ckpt_src.name != 'ckpt.pt':
        raise ValueError('Argument base_model musi wskazywac katalog modelu albo plik ckpt.pt.')
    if not ckpt_src.exists():
        raise FileNotFoundError(f'Nie znaleziono checkpointu: {ckpt_src}')
    source_out_dir = ckpt_src.parent

    import torch

    checkpoint = torch.load(ckpt_src, map_location='cpu')
    checkpoint_cfg = checkpoint.get('config', {})
    checkpoint_model_args = checkpoint.get('model_args', {})
    checkpoint_dataset = checkpoint_cfg.get('dataset')
    checkpoint_block_size = checkpoint_model_args.get('block_size')
    checkpoint_vocab_size = checkpoint_model_args.get('vocab_size')

    if checkpoint_dataset is None:
        raise ValueError('Checkpoint nie zawiera nazwy datasetu w config[\'dataset\'].')

    source_meta_path = pkg_root / 'data' / checkpoint_dataset / 'meta.pkl'
    if not source_meta_path.exists():
        raise FileNotFoundError(f'Nie znaleziono tokenizera modelu: {source_meta_path}')

    with open(source_meta_path, 'rb') as f:
        source_meta = pickle.load(f)

    stoi = source_meta.get('stoi')
    itos = source_meta.get('itos')
    vocab_size = source_meta.get('vocab_size')
    if not isinstance(stoi, dict) or not isinstance(itos, dict):
        raise ValueError('Niepoprawny format source meta.pkl (brak stoi/itos).')

    if checkpoint_vocab_size is not None and vocab_size != checkpoint_vocab_size:
        raise ValueError(
            'Niezgodny vocab_size miedzy checkpointem a tokenizerem: '
            f'{checkpoint_vocab_size} != {vocab_size}'
        )

    text = src.read_text(encoding='utf-8')
    if len(text) < 2:
        raise ValueError('Plik wejsciowy jest zbyt krotki do podzialu train/val.')

    unknown_chars = sorted({ch for ch in text if ch not in stoi})
    if unknown_chars:
        preview = ''.join(unknown_chars[:20])
        raise ValueError(
            f'Tekst zawiera {len(unknown_chars)} znakow spoza tokenizera modelu. '
            f'Przyklad: {preview!r}'
        )

    ids = np.array([stoi[ch] for ch in text], dtype=np.uint16)
    split_idx = int(len(ids) * train_split)
    split_idx = max(1, min(len(ids) - 1, split_idx))
    train_ids = ids[:split_idx]
    val_ids = ids[split_idx:]

    if dataset_name is None:
        base_dataset = re.sub(r'[^0-9A-Za-z_]+', '_', src.stem).strip('_') or 'custom_char'
        dataset_name = f'{base_dataset}_finetune'

    data_dir = pkg_root / 'data' / dataset_name
    data_dir.mkdir(parents=True, exist_ok=True)
    input_txt = data_dir / 'input.txt'
    shutil.copy2(src, input_txt)
    train_ids.tofile(data_dir / 'train.bin')
    val_ids.tofile(data_dir / 'val.bin')

    meta = {
        'vocab_size': vocab_size,
        'itos': itos,
        'stoi': stoi,
    }
    with open(data_dir / 'meta.pkl', 'wb') as f:
        pickle.dump(meta, f)

    if out_dir is None:
        model_out_dir = source_out_dir
    else:
        model_out_dir = Path(out_dir)
        if not model_out_dir.is_absolute():
            model_out_dir = pkg_root / model_out_dir
    model_out_dir.mkdir(parents=True, exist_ok=True)

    ckpt_dst = model_out_dir / 'ckpt.pt'
    if ckpt_dst.resolve() != ckpt_src.resolve():
        shutil.copy2(ckpt_src, ckpt_dst)

    if block_size is None:
        block_size = checkpoint_block_size
    if block_size is None:
        raise ValueError('Nie mozna ustalic block_size z checkpointu. Podaj block_size recznie.')
    if checkpoint_block_size is not None and block_size > checkpoint_block_size:
        raise ValueError(
            f'block_size={block_size} nie moze byc wiekszy niz block_size modelu ({checkpoint_block_size}).'
        )

    if device is None:
        device = 'cuda' if shutil.which('nvidia-smi') else 'cpu'

    print(f'Plik wejsciowy: {src.resolve()}')
    print(f'Checkpoint bazowy: {ckpt_src.resolve()}')
    print(f'Model bedzie zapisany w: {model_out_dir.resolve()}')
    print(f'Dataset nanoGPT: {dataset_name}')

    cmd = [
        sys.executable,
        'train.py',
        f'--dataset={dataset_name}',
        f'--out_dir={model_out_dir}',
        '--init_from=resume',
        f'--max_iters={max_iters}',
        f'--eval_interval={eval_interval}',
        f'--log_interval={log_interval}',
        f'--batch_size={batch_size}',
        f'--block_size={block_size}',
        f'--learning_rate={learning_rate}',
        f'--device={device}',
        f'--compile={compile_model}',
        '--always_save_checkpoint=True',
    ]

    process = subprocess.Popen(
        cmd,
        cwd=str(pkg_root),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    if process.stdout is None:
        raise RuntimeError('Nie udalo sie przechwycic logow treningu.')

    for line in process.stdout:
        print(line, end='')

    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f'Dotrenowanie nanoGPT zakonczyl sie kodem {return_code}.')

    return model_out_dir


# Przyklad uruchomienia:
'''

train_nanogpt_on_file(
    "NEW_CORPUS_SMASH.txt",
    device="cuda",
    n_layer=24,
    n_head=24,
    n_embd=768,
    batch_size=12,
    block_size=1024,
    max_iters=5000,
    eval_interval=200,
    learning_rate=0.01
)


finetune_nanogpt_on_file(
    "NEW_CORPUS_PROVING.txt",
    "out-NEW_CORPUS_PROVING",
    device="cuda",
    batch_size=12,
    block_size=1024,
    max_iters=10000,
    eval_interval=20,
    learning_rate=0.00001
)
'''

'\n\ntrain_nanogpt_on_file(\n    "NEW_CORPUS_SMASH.txt",\n    device="cuda",\n    n_layer=24,\n    n_head=24,\n    n_embd=768,\n    batch_size=12,\n    block_size=1024,\n    max_iters=5000,\n    eval_interval=200,\n    learning_rate=0.01\n)\n\n\nfinetune_nanogpt_on_file(\n    "NEW_CORPUS_PROVING.txt",\n    "out-NEW_CORPUS_PROVING",\n    device="cuda",\n    batch_size=12,\n    block_size=1024,\n    max_iters=10000,\n    eval_interval=20,\n    learning_rate=0.00001\n)\n'

In [64]:
def FreeVariablesFormula(x):
    if not isinstance(x,Formula):
        raise TypeError("Not Formula")
    if isinstance(x,Truth) or isinstance(x,Lie):
        return []
    elif isinstance(x,Atom):
        return [x]
    else:
        ans = []
        for i in x.Interior:
            ans = list(set(ans).union(set(FreeVariablesFormula(i))))
        return ans

In [65]:
def FreeVariablesLP(x):
    if not isinstance(x, LittleProblem):
        raise TypeError("Bad argument")
    ans = FreeVariablesFormula(x.conclusion)
    for i in x.assumptions.base_context:
        ans = list(set(ans).union(set(FreeVariablesFormula(i))))
    for i in x.assumptions.additional_context:
        ans = list(set(ans).union(set(FreeVariablesFormula(i))))
    return ans




In [66]:
FreeVariablesFormula(parse_infix('x ∨ (x ∨ z)'))

[R0(z), R0(x)]

In [67]:

'''
text = NEW_CORPUS_PROVING
text = text.split("\n\n")
text = text[:-1]
'''
def goal_from_proof_with_assumption_text(text_proof):
    proof = text_proof.splitlines()
    thesis_conclusion = parse_infix(proof[0][12:])
    base_context = []
    if proof[1][0] == "a":
        if proof[1][-1] == ",":
            base_context.append(parse_infix(proof[1][15:-1]))
            j = 2
            while proof[j][0] != "1":
                if proof[j][-1] == ",":
                    base_context.append(parse_infix(proof[j][:-1]))
                else:
                    base_context.append(parse_infix(proof[j]))
                j += 1
        else:
            base_context.append(parse_infix(proof[1][15:]))

    return LittleProblem(Context(base_context,[]),thesis_conclusion)
'''
max = 0
s = 0
def frac(i):
    if i == 0:
        return 1
    else:
        return frac(i-1) * i
for i in tqdm(range(len(text))):
    goal = goal_from_proof_with_assumption_text(text[i])
    if len(FreeVariablesLP(goal)) - len(FreeVariablesFormula(goal.conclusion)) > max:
        max = len(FreeVariablesLP(goal)) - len(FreeVariablesFormula(goal.conclusion))
    if len(FreeVariablesLP(goal)) - len(FreeVariablesFormula(goal.conclusion)) >5:
        print(1)
    s += frac(len(FreeVariablesLP(goal)) - len(FreeVariablesFormula(goal.conclusion)))
print(s/len(text))
'''

'\nmax = 0\ns = 0\ndef frac(i):\n    if i == 0:\n        return 1\n    else:\n        return frac(i-1) * i\nfor i in tqdm(range(len(text))):\n    goal = goal_from_proof_with_assumption_text(text[i])\n    if len(FreeVariablesLP(goal)) - len(FreeVariablesFormula(goal.conclusion)) > max:\n        max = len(FreeVariablesLP(goal)) - len(FreeVariablesFormula(goal.conclusion))\n    if len(FreeVariablesLP(goal)) - len(FreeVariablesFormula(goal.conclusion)) >5:\n        print(1)\n    s += frac(len(FreeVariablesLP(goal)) - len(FreeVariablesFormula(goal.conclusion)))\nprint(s/len(text))\n'

In [68]:
def AbstractionSimple(t,x,Expr):
    if t == Expr:
        return parse_infix(x)
    if isinstance(Expr,Truth) or isinstance(Expr,Lie) or isinstance(Expr,Atom):
        return Expr
    ExprAns = copy.deepcopy(Expr)
    for i in range(Expr.Interior.__len__()):
        ExprAns.Interior[i] = AbstractionSimple(t,x,ExprAns.Interior[i])
    return ExprAns

def normalise(f):
    vars = variables_order(f)
    i = 0
    new_vars = dict()
    for x in vars:
        if not to_infix(x) in new_vars.keys():
            new_vars[to_infix(x)] = "x" + str(i)
            i += 1
    ans = copy.deepcopy(f)
    for old in new_vars.keys():
        new = new_vars[old]
        ans = AbstractionSimple(parse_infix(old), new, ans)
    return ans



In [69]:
PROJECT_ROOT = Path.home() / "PycharmProjects" / "Magisterka"
PKG_ROOT = PROJECT_ROOT / "nanoGPT-20251228T135841Z-3-001"
sys.path.insert(0, str(PKG_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))
print(sys.path)
sys.path.insert(0, str(PROJECT_ROOT))
from nanoGPT.model import GPTConfig, GPT
print(GPT)
from dataclasses import dataclass
from typing import Callable
import torch
@dataclass
class NanoGPTBundle:
    model: GPT
    encode: Callable[[str], list[int]]
    device: str
    device_type: str
    ptdtype: torch.dtype

from dataclasses import dataclass
from typing import Callable, Optional
import os
import pickle
import torch
import tiktoken



@dataclass
class NanoGPTBundle:
    model: GPT
    encode: Callable[[str], list[int]]
    decode_token: Callable[[int], str]         # <-- NOWE: id -> tekst tokenu
    decode_ids: Callable[[list[int]], str]     # <-- NOWE: lista id -> tekst
    device: str
    device_type: str
    ptdtype: torch.dtype


def load_nanogpt_bundle(
    out_dir: str,
    *,
    device: Optional[str] = None,
    dtype: Optional[str] = None,
) -> NanoGPTBundle:
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    requested_device = device
    requested_device_type = "cuda" if "cuda" in requested_device else "cpu"

    if dtype is None:
        if requested_device_type == "cuda" and torch.cuda.is_bf16_supported():
            dtype = "bfloat16"
        elif requested_device_type == "cuda":
            dtype = "float16"
        else:
            dtype = "float32"

    ptdtype = {"float32": torch.float32, "float16": torch.float16, "bfloat16": torch.bfloat16}[dtype]
    project_root = os.path.abspath(globals().get("PROJECT_ROOT", os.getcwd()))
    raw_pkg_roots = [
        globals().get("PKG_ROOT"),
        os.path.join(project_root, "nanoGPT-20251228T135841Z-3-001", "nanoGPT"),
        os.path.join(project_root, "nanoGPT"),
    ]
    pkg_roots = []
    for root in raw_pkg_roots:
        if not root:
            continue
        root_abs = os.path.abspath(os.path.expanduser(str(root)))
        if root_abs in pkg_roots:
            continue
        if os.path.exists(os.path.join(root_abs, "train.py")):
            pkg_roots.append(root_abs)
    if not pkg_roots:
        raise FileNotFoundError("Nie znaleziono katalogu nanoGPT z train.py.")

    out_dir_path = os.path.expanduser(out_dir)
    ckpt_candidates = []
    if os.path.isabs(out_dir_path):
        ckpt_candidates.append(out_dir_path if out_dir_path.endswith("ckpt.pt") else os.path.join(out_dir_path, "ckpt.pt"))
    else:
        for root in pkg_roots:
            base = os.path.join(root, out_dir_path)
            ckpt_candidates.append(base if out_dir_path.endswith("ckpt.pt") else os.path.join(base, "ckpt.pt"))
        project_base = os.path.join(project_root, out_dir_path)
        ckpt_candidates.append(project_base if out_dir_path.endswith("ckpt.pt") else os.path.join(project_base, "ckpt.pt"))

    ckpt_path = next((p for p in ckpt_candidates if os.path.exists(p)), None)
    if ckpt_path is None:
        raise FileNotFoundError(
            "Nie znaleziono checkpointu. Sprawdzane sciezki: " + "; ".join(ckpt_candidates)
        )

    real_ckpt_path = os.path.realpath(ckpt_path)
    ckpt_pkg_root = next(
        (root for root in pkg_roots if os.path.commonpath([real_ckpt_path, os.path.realpath(root)]) == os.path.realpath(root)),
        None,
    )
    if ckpt_pkg_root is None:
        inferred_root = os.path.dirname(os.path.dirname(real_ckpt_path))
        if os.path.exists(os.path.join(inferred_root, "train.py")):
            ckpt_pkg_root = inferred_root
        else:
            ckpt_pkg_root = pkg_roots[0]

    checkpoint = torch.load(real_ckpt_path, map_location="cpu", weights_only=False)

    gptconf = GPTConfig(**checkpoint["model_args"])
    model = GPT(gptconf)

    state_dict = checkpoint["model"]
    unwanted_prefix = "_orig_mod."
    for k in list(state_dict.keys()):
        if k.startswith(unwanted_prefix):
            state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)

    model.load_state_dict(state_dict)
    active_device = requested_device
    active_device_type = requested_device_type
    active_ptdtype = ptdtype
    try:
        model.eval().to(active_device)
    except Exception as e:
        msg = str(e)
        if requested_device_type == "cuda" and ("CUDA error" in msg or "device-side assert triggered" in msg):
            print("Uwaga: CUDA jest w blednym stanie. Przelaczam model na CPU. Zrestartuj kernel, aby wrocic na CUDA.")
            active_device = "cpu"
            active_device_type = "cpu"
            active_ptdtype = torch.float32
            model.eval().to(active_device)
        else:
            raise

    # --- encoder/decoder ---
    encode = None
    decode_token = None
    decode_ids = None

    dataset = checkpoint.get("config", {}).get("dataset")
    model_vocab_size = checkpoint.get("model_args", {}).get("vocab_size")
    if dataset:
        meta_path = os.path.join(ckpt_pkg_root, "data", dataset, "meta.pkl")
        if os.path.exists(meta_path):
            with open(meta_path, "rb") as f:
                meta = pickle.load(f)
            stoi, itos = meta["stoi"], meta["itos"]

            def encode(s: str) -> list[int]:
                return [stoi[c] for c in s]

            def decode_token(i: int) -> str:
                return itos[i]

            def decode_ids(ids: list[int]) -> str:
                return "".join(itos[i] for i in ids)

    if encode is None:
        if model_vocab_size not in (50257, 50304):
            raise FileNotFoundError(
                f"Nie znaleziono meta.pkl dla datasetu={dataset!r} (oczekiwano w {os.path.join(ckpt_pkg_root, 'data', str(dataset), 'meta.pkl')}). "
                f"Model ma vocab_size={model_vocab_size}, wiec fallback do tiktoken (GPT-2) bylby niepoprawny."
            )
        enc = tiktoken.get_encoding("gpt2")

        def encode(s: str) -> list[int]:
            return enc.encode(s, allowed_special={"<|endoftext|>"})

        def decode_token(i: int) -> str:
            return enc.decode([i])

        def decode_ids(ids: list[int]) -> str:
            return enc.decode(ids)

    return NanoGPTBundle(
        model=model,
        encode=encode,
        decode_token=decode_token,
        decode_ids=decode_ids,
        device=active_device,
        device_type=active_device_type,
        ptdtype=active_ptdtype,
    )

bundle_proving = load_nanogpt_bundle("out-NEW_CORPUS_PROVING")
bundle_smashing = load_nanogpt_bundle("out-NEW_CORPUS_SMASH")


['/home/lukasz/PycharmProjects/Magisterka', '/home/lukasz/PycharmProjects/Magisterka/nanoGPT-20251228T135841Z-3-001', '/home/lukasz/PycharmProjects/Magisterka/nanoGPT-20251228T135841Z-3-001/nanoGPT', '/home/lukasz/PycharmProjects/Magisterka/nanoGPT-20251228T135841Z-3-001/nanoGPT', '/home/lukasz/PycharmProjects/Magisterka/nanoGPT-20251228T135841Z-3-001/nanoGPT', '/home/lukasz/PycharmProjects/Magisterka/nanoGPT-20251228T135841Z-3-001/nanoGPT', '/home/lukasz/PycharmProjects/Magisterka/nanoGPT-20251228T135841Z-3-001/nanoGPT', '/snap/pycharm/35/plugins/python-ce/helpers/jupyter_debug', '/snap/pycharm/35/plugins/python-ce/helpers/pydev', '/home/lukasz/PycharmProjects/Magisterka', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/home/lukasz/PycharmProjects/Magisterka/.venv/lib/python3.12/site-packages']
<class 'nanoGPT.model.GPT'>
number of parameters: 169.95M
number of parameters: 169.95M


In [70]:
import numpy as np
import torch
from typing import Optional
from tqdm import tqdm


@torch.no_grad()
def next_token_distribution_from_bundle(
    bundle: NanoGPTBundle,
    prompt: str,
    *,
    temperature: float = 1.0,
    top_k: Optional[int] = None,
) -> np.ndarray:
    x = torch.tensor(bundle.encode(prompt), dtype=torch.long, device=bundle.device)[None, :]

    with torch.autocast(
        device_type=bundle.device_type,
        dtype=bundle.ptdtype,
        enabled=(bundle.device_type == "cuda" and bundle.ptdtype != torch.float32),
    ):
        logits, _ = bundle.model(x)

    # logits: [1, T, V]
    next_logits = logits[0, -1, :] / temperature

    if top_k is not None:
        values, indices = torch.topk(next_logits, top_k)
        probs = torch.softmax(values, dim=-1)  # <-- TU powstają prawdopodobieństwa (dla top_k)
        return np.array([(int(i), float(p)) for i, p in zip(indices, probs)], dtype=float)

    probs = torch.softmax(next_logits, dim=-1)  # <-- TU powstają prawdopodobieństwa (dla całego vocab)
    return np.array([(int(i), float(probs[i])) for i in range(probs.shape[0])], dtype=float)



def random_next_token(
        bundle: NanoGPTBundle,
        prompt: str,
        verbose: bool = False,
        temperature: float = 1.0,
        randomize: float = 0.0

):
    source = next_token_distribution_from_bundle(bundle,prompt,temperature=temperature)
    token_ids = source[:,0]
    probs = source[:,1]
    if verbose:
        verbose_table = []
        for i in range(len(token_ids)):
            verbose_table.append([bundle.decode_token(token_ids[i]),probs[i]])
        verbose_table.sort(key=lambda x: x[1],reverse=True)
        for i in verbose_table[:10]:
            print(i)
    return bundle.decode_token(random.choices(token_ids,weights=probs)[0])

prompt = "Prove that: (a ∧ b"


def last_token_probability_from_bundle(
    bundle: NanoGPTBundle,
    prompt: str,
    *,
    temperature: float = 1.0,
    top_k: Optional[int] = None,
):
    distribution = next_token_distribution_from_bundle(bundle, prompt[:-1], temperature=temperature, top_k=top_k)
    key = bundle.encode(prompt[-1])[0]
    return distribution[key][1]
prompt = "ab"
print(last_token_probability_from_bundle(bundle_proving,prompt))
print(last_token_probability_from_bundle(bundle_smashing,prompt))

4.805624485015869e-07
2.1010637283325195e-06


In [71]:
def next_line(
    bundle: NanoGPTBundle,
    prompt: str,
    *,
    temperature: float = 1.0,
    temperature_begin_line : float = 10,
    multiplier_begin_line : float = 0.66666666,
    top_k: Optional[int] = None
)->str:
    ans = prompt
    next_token = None
    while next_token != "\n":
        next_token = random_next_token(bundle,ans,temperature=temperature)
        ans += next_token
    return ans.splitlines()[-1]
prompt = '''Smash:  ⊢ (a ∧ b) → (¬a → c)
'''
prompt += next_line(bundle_smashing,prompt,temperature=1)
prompt += "\n"
prompt += next_line(bundle_smashing,prompt,temperature=1)
print(prompt)

Smash:  ⊢ (a ∧ b) → (¬a → c)
With: ImplicationIntroduction
To: a ∧ b ⊢ ¬a → c


In [72]:
def is_ready_to_check_smash(prompt):
    if not isinstance(prompt,str):
        raise TypeError("Bad argument")
    prompt = prompt.rstrip("\n")
    prompt = prompt.splitlines()
    if len(prompt) < 3:
        return False
    else:
        return prompt[-1][-1] != ","

prompt = '''Smash: ¬¬k ⊢ k ∧ k
With: ConjunctionIntroduction
To: ¬¬k ⊢ k,
¬¬k ⊢ k'''
is_ready_to_check_smash(prompt)
prompt

'Smash: ¬¬k ⊢ k ∧ k\nWith: ConjunctionIntroduction\nTo: ¬¬k ⊢ k,\n¬¬k ⊢ k'

In [73]:
def silliest_generate_smash(bundle : NanoGPTBundle,
                           prompt : str):
    c = True
    while c:
        while not is_ready_to_check_smash(prompt):
            prompt += next_line(bundle,prompt)
            prompt += "\n"
        try:
            ans_structure = check_smash_from_text(prompt)
            acceptable_variables = free_variables_LP(ans_structure.problem)
            for i in ans_structure.subproblems:
                vars = free_variables_LP(i)
                for j in vars:
                    if not j in acceptable_variables:
                        raise Exception("Bad variable")
            c = False
        except Exception:
            prompt = prompt.splitlines()[0]+ "\n"

    return prompt
prompt = '''Smash:  ⊢ (a ∧ b) → (¬a → c)
'''

silliest_generate_smash(bundle_smashing,prompt)

'Smash:  ⊢ (a ∧ b) → (¬a → c)\nWith: ImplicationIntroduction\nTo: a ∧ b ⊢ ¬a → c\n'

In [74]:
def silliest_generate_proof(bundle : NanoGPTBundle,
                            prompt : str,
                            temperature : float = 1.0,
                            max_new_lines : int = 1,
                            max_failed_attempts : int = 200):
    prompt_str = prompt.rstrip("\n") + "\n"  #dodalem
    prompt = [line for line in prompt_str.splitlines() if line.strip() != ""]  #dodalem
    thesis_conclusion = parse_infix(prompt[0][12:])
    base_context = []
    if len(prompt) > 1:
        if prompt[1][0] == "a":
            if prompt[1][-1] == ",":
                base_context.append(parse_infix(prompt[1][15:-1]))
                j = 2
                while j < len(prompt) and prompt[j][0] != "1":
                    if prompt[j][-1] == ",":
                        base_context.append(parse_infix(prompt[j][:-1]))
                    else:
                        base_context.append(parse_infix(prompt[j]))
                    j += 1
            else:
                base_context.append(parse_infix(prompt[1][15:]))
    goal = LittleProblem(Context(base_context,[]) , thesis_conclusion)
    acceptable_variables =FreeVariablesLP(goal)
    existing_proof = ""  #dodalem
    for i in range(len(goal.assumptions.base_context)+1,len(prompt)):
        existing_proof += prompt[i] + "\n"  #dodalem
    added_proof = ""  #dodalem
    last_parsed_line = None
    new_lines = 0
    failed_attempts = 0
    while last_parsed_line != goal and new_lines < max_new_lines:
        nl =  next_line(bundle,prompt_str+added_proof,temperature=temperature)  #dodalem
        failed_attempts += 1
        if failed_attempts > max_failed_attempts:
            break
        stripped_nl = nl.strip()
        if stripped_nl == "":
            continue
        if "." not in stripped_nl or not stripped_nl.split(".", 1)[0].isdigit():
            continue
        try:
            temporary_added_proof = added_proof + nl + "\n"  #dodalem
            temporary_proof = existing_proof + temporary_added_proof  #dodalem
            parsedProof = parseProof(temporary_proof[:-1],base_context)
            last_parsed_line = parsedProof[len(parsedProof)].LP
            last_parsed_variables = FreeVariablesLP(last_parsed_line)
            for i in last_parsed_variables:
                if i not in acceptable_variables:
                    raise Exception("Nieprawidlowe zmienne w tym wierszu")
            #print(temporary_proof)
            added_proof = temporary_added_proof  #dodalem
            new_lines += 1
            failed_attempts = 0
        except Exception:
            pass
    return prompt_str+added_proof  #dodalem


prompt = '''Prove that: (a ∧ b) → (¬a → c)
1. ¬a    assumption
'''
silliest_generate_proof(bundle_proving,prompt,temperature=0.1,max_new_lines=4)


'Prove that: (a ∧ b) → (¬a → c)\n1. ¬a    assumption\n2. a ∧ b    assumption\n3. a   ∧-elimination1, 2\n4. ⊥   ¬-elimination, 1, 3\n5. ¬¬a   ¬-introduction, 1–4\n'

In [75]:
def prompt_probability_normalised(bundle, prompt,temperature=1,top_k=None):
    temporal_prompt = prompt.splitlines()[0] + "\n"
    ans_unnormalised = 0
    proof = ""
    lines_of_goal = 1
    for i in range(len(prompt.splitlines())):
        if prompt.splitlines()[i][0] == "1":
            lines_of_goal  = i
    for i in prompt.splitlines()[lines_of_goal:]:
        proof += i+"\n"
    for i in proof:
        temporal_prompt += i
        #print(temporal_prompt)
        ans_unnormalised  += np.log(last_token_probability_from_bundle(bundle,temporal_prompt,temperature=temperature,top_k=top_k))
        #print(ans_unnormalised)
    return ans_unnormalised/len(proof)

X = silliest_generate_proof(bundle_proving,prompt,temperature=0.1,max_new_lines=1)

print(prompt_probability_normalised(bundle_proving,X))

def soft_max(x):
    return np.exp(x)/np.sum(np.exp(x))

soft_max([1,2])

-0.04602193000684427


array([0.26894142, 0.73105858])

In [76]:
import re

def int_from_start(s: str):
    m = re.match(r'\d+', s)
    return int(m.group()) if m else None

def is_well_numbered(proof:str):
    #print("^^^^^^^^^^^^^^^^^^",proof,"___________________________")
    for i in range(len(proof.splitlines())):
        if int_from_start(proof.splitlines()[i]) != i+1:
            raise Exception("Nieprawidlowe numerywanie wierszy")
    return True

In [77]:
def find_proof_little_smarter(
    bundle: NanoGPTBundle,
    prompt: str,
    *,
    temperature: float = 1.0,
    max_iter: int = 10000
):
    def extract_proof_part(full_prompt: str):  #dodalem
        lines = [line for line in full_prompt.splitlines() if line.strip() != ""]  #dodalem
        first_proof_idx = None  #dodalem
        for idx, line in enumerate(lines):  #dodalem
            if int_from_start(line) == 1:  #dodalem
                first_proof_idx = idx  #dodalem
                break  #dodalem
        if first_proof_idx is None:  #dodalem
            return ""  #dodalem
        return "\n".join(lines[first_proof_idx:]) + "\n"  #dodalem
    prompt_str = prompt.rstrip("\n") + "\n"  #dodalem
    prompt = [line for line in prompt_str.splitlines() if line.strip() != ""]  #dodalem
    thesis_conclusion = parse_infix(prompt[0][12:])
    base_context = []
    if len(prompt) > 1:
        if prompt[1][0] == "a":
            if prompt[1][-1] == ",":
                base_context.append(parse_infix(prompt[1][15:-1]))
                j = 2
                while j < len(prompt) and prompt[j][0] != "1":
                    if prompt[j][-1] == ",":
                        base_context.append(parse_infix(prompt[j][:-1]))
                    else:
                        base_context.append(parse_infix(prompt[j]))
                    j += 1
            else:
                base_context.append(parse_infix(prompt[1][15:]))
    goal = LittleProblem(Context(base_context,[]) , thesis_conclusion)
    acceptable_variables =FreeVariablesLP(goal)
    prompts = [prompt_str]
    probabilities_wild = [1.0]
    probabilities = np.array(probabilities_wild)
    I = 0
    while I < max_iter:
        prompt_str = random.choices(prompts,weights=probabilities)[0]
        prompt_str = prompt_str.rstrip("\n") + "\n"  #dodalem
        I += 1
        print(I)
        try:
            new_prompt_str = silliest_generate_proof(bundle,prompt_str,temperature=temperature,max_new_lines=1)
            print(new_prompt_str)
            new_proof = extract_proof_part(new_prompt_str)  #dodalem
            if new_proof == "":  #dodalem
                raise Exception("Brak linii dowodu")  #dodalem
            print(new_proof)  #dodalem
            #print(new_proof)
            is_well_numbered(new_proof[:-1])

            #print("ok")
            parsedProof = parseProof(new_proof[:-1],base_context)
            if parsedProof[len(parsedProof)].LP == goal:
                return new_prompt_str
            elif not new_prompt_str in prompts:
                probabilities_wild.append(prompt_probability_normalised(bundle,new_prompt_str))
                def mean(x):
                    ans = 0
                    for i in x:
                        ans += i
                    return ans/len(x)
                probabilities_wild[0] = mean(probabilities_wild[1:])
                probabilities = soft_max(probabilities_wild)
                prompts.append(new_prompt_str)
        except Exception as e:
            print("[find_proof_little_smarter]", type(e).__name__, e)
    return prompts,probabilities
prompt = "Prove that: (a ∧ b) → (¬a → c)"

#ans = find_proof_little_smarter(bundle_proving,prompt,max_iter=20)

In [78]:
prompt = '''Prove that: (a ∧ b) → (¬a → c)
1. a ∧ b   assumption
2. a    ∧-elimination1, 1
'''
silliest_generate_proof(bundle_proving,prompt)

'Prove that: (a ∧ b) → (¬a → c)\n1. a ∧ b   assumption\n2. a    ∧-elimination1, 1\n3. ¬a   assumption\n'

In [79]:
prompt = '''Smash: v ∨ g ⊢ g ∨ v
With: DisjunctionElimination
'''
x = silliest_generate_smash(bundle_smashing,prompt)
print(x)
print(check_smash_from_text(x))


Smash: v ∨ g ⊢ g ∨ v
With: ImplicationElimination
To:  ⊢ (v ∨ g) → (g ∨ v),
v ∨ g ⊢ v ∨ g

v ∨ g  ⊢ g ∨ v [  ⊢ (v ∨ g) → (g ∨ v) . v ∨ g  ⊢ v ∨ g] ImplicationElimination


In [80]:
def prompts_to_prove_from_smashed_problem(SP : smashed_problem):
    ANS = []
    for i in SP.subproblems:
        ans = "Prove that: " + to_infix(i.conclusion) + "\n"
        if i.assumptions.base_context != []:
            print(i.assumptions.base_context)
            ans += "assuming that: "
            for j in range(len(i.assumptions.base_context)-1,-1,-1):
                ans += to_infix(i.assumptions.base_context[j]) + ",\n"
            ans = ans[:-2] + "\n"
        ANS.append(ans)
    return ANS

def prompt_to_prove_from_little_problem(LP : LittleProblem):
    ans = "Prove that: " + to_infix(LP.conclusion) + "\n"
    if LP.assumptions.base_context != []:
        print(LP.assumptions.base_context)
        ans += "assuming that: "
        for j in range(len(LP.assumptions.base_context)-1,-1,-1):
            ans += to_infix(LP.assumptions.base_context[j]) + ",\n"
        ans = ans[:-2] + "\n"
    if LP.assumptions.additional_context != []:
        raise Exception("Nieprawidlowe dodatkowe konteksty")
    return ans

smashed = prompts_to_prove_from_smashed_problem(check_smash_from_text(x))
for i in smashed:
    print(i)


[R0(v) ∨ R0(g)]
Prove that: (v ∨ g) → (g ∨ v)

Prove that: v ∨ g
assuming that: v ∨ g



In [81]:
print(prompt_to_prove_from_little_problem(LittleProblem(Context([parse_infix("x"),parse_infix("y")],[]),parse_infix("z"))))

[R0(x), R0(y)]
Prove that: z
assuming that: y,
x



In [82]:
problem1 = prompts_to_prove_from_smashed_problem(check_smash_from_text(silliest_generate_smash(bundle_smashing,"Smash: ⊢ (a ∧ b) → (¬a → c)\n")))[0]
print(check_smash_from_text(silliest_generate_smash(bundle_smashing,"Smash: ⊢ (a ∧ b) → (¬a → c)\n")))
problem2 = prompts_to_prove_from_smashed_problem(check_smash_from_text(silliest_generate_smash(bundle_smashing,"Smash: a ∧ b  ⊢ ¬a → c\n")))[0]
smashings = []
for i in range(10):
    problem3 = prompts_to_prove_from_smashed_problem(check_smash_from_text(silliest_generate_smash(bundle_smashing,"Smash: a ∧ b, ¬a ⊢ c\n")))
    if not problem3 in smashings:
        smashings.append(problem3)
#    print(problem3)
#problem3 = "Prove that: ⊥\nassuming that: a ∧ b,\n¬a,\n¬c\n"
#find_proof_little_smarter(bundle_proving,problem3)

print(smashings)

[R0(a) ∧ R0(b)]
  ⊢ (a ∧ b) → (¬a → c) [a ∧ b  ⊢ ¬a → c] ImplicationIntroduction
[¬R0(a), R0(a) ∧ R0(b)]
[R0(a) ∧ R0(b), ¬R0(a)]
[R0(a) ∧ R0(b), ¬R0(a)]
[R0(a) ∧ R0(b), ¬R0(a)]
[R0(a) ∧ R0(b), ¬R0(a)]
[R0(a) ∧ R0(b), ¬R0(a), ¬R0(c)]
[R0(a) ∧ R0(b), ¬R0(a), ¬R0(c)]
[R0(a) ∧ R0(b), ¬R0(a)]
[R0(a) ∧ R0(b), ¬R0(a)]
[R0(a) ∧ R0(b), ¬R0(a)]
[R0(a) ∧ R0(b), ¬R0(a), ¬R0(c)]
[['Prove that: c ∧ ⊥\nassuming that: ¬a,\na ∧ b\n'], ['Prove that: ¬¬c\nassuming that: ¬a,\na ∧ b\n'], ['Prove that: c ∧ c\nassuming that: ¬a,\na ∧ b\n'], ['Prove that: ⊥\nassuming that: ¬c,\n¬a,\na ∧ b\n']]


In [83]:
for i in smashings:
    print(i)

['Prove that: c ∧ ⊥\nassuming that: ¬a,\na ∧ b\n']
['Prove that: ¬¬c\nassuming that: ¬a,\na ∧ b\n']
['Prove that: c ∧ c\nassuming that: ¬a,\na ∧ b\n']
['Prove that: ⊥\nassuming that: ¬c,\n¬a,\na ∧ b\n']


In [84]:
class node:
    def __init__(self,
                 Interior,
                 Parent = None,
                 Children = [],
                 Text : str = "",
                 Value : bool = False,
                 Probability: float = 1.0):
        if not (isinstance(Interior,smashed_problem) or isinstance(Interior,LittleProblem)):
            raise TypeError("Bad argument")
        if isinstance(Interior,smashed_problem):
            if not (Parent == None):
                if isinstance(Parent,node):
                    if not isinstance(Parent.Interior,LittleProblem):
                        raise TypeError("Bad argument")
                else:
                    raise TypeError("Bad argument")
            for i in Children:
                if not isinstance(i,node):
                    raise TypeError("Bad argument")
                elif not isinstance(i.Interior,LittleProblem):
                    raise TypeError("Bad argument")
        elif isinstance(Interior,LittleProblem):
            if not (Parent == None):
                if isinstance(Parent,node):
                    if not isinstance(Parent.Interior,smashed_problem):
                        raise TypeError("Bad argument")
                else:
                    raise TypeError("Bad argument")
            for i in Children:
                if not isinstance(i,node):
                    raise TypeError("Bad argument")
                elif not isinstance(i.Interior,smashed_problem):
                    raise TypeError("Bad argument")
        else:
            raise TypeError("Bad argument")

        self.Interior = Interior
        self.Parent = Parent
        self.Children = Children
        if Text == None:
            if isinstance(Interior,LittleProblem):
                Text = prompt_to_prove_from_little_problem(Interior)
        self.Text = Text
        self.Value = Value
        self.Probability = Probability
    def Smash_Leaf(self,
                   bundle_smashing : NanoGPTBundle,
                   deg : int = 1):
        if isinstance(self.Interior,smashed_problem):
            raise Exception("not LP")
        if self.Children != []:
            raise Exception("not leaf")
        prompt = "Smash: " + to_infix_LP(self.Interior) + "\n"
        weights = dict()
        for i in range(deg):
            SPText  = silliest_generate_smash(bundle_smashing,prompt)
            if len(SPText.splitlines()) != 1:
                if SPText in weights.keys():
                    weights[SPText] += 1
                else:
                    weights[SPText] = 1
        for i in weights.keys():
            self.Children.append(
                node(
                check_smash_from_text(i),
                Parent=self,
                Children=[],
                Text=i,
                Value=False,
                Probability=weights[i]/deg
            ))
        for i in self.Children:
            grand_children_interiors = i.Interior.subproblems
            for j in grand_children_interiors:
                i.Children.append(
                    node(
                    j,
                    Parent=i,
                    Children=[],
                    Text=prompt_to_prove_from_little_problem(j),
                    Value=False,
                    Probability=1.0
                ))
            if i.Children == []:
                i.Value = True
                ancestor = i.Parent
                while ancestor != None:
                    print("ok")
                    if isinstance(ancestor.Interior,smashed_problem):
                        value = True
                        for j in ancestor.Children:
                            value = value and j.Value
                        ancestor.Value = value
                    if isinstance(ancestor.Interior,LittleProblem):
                        value = False
                        for j in ancestor.Children:
                            value = value or j.Value
                        ancestor.Value = value
                    ancestor = ancestor.Parent

    def RunLeaf(self,
                bundle_proving : NanoGPTBundle,
                bundle_smashing : NanoGPTBundle,
                temperature : float = 1.0,
                max_iter : int = 20,
                deg : int = 1):
        if isinstance(self.Interior,smashed_problem):
            raise Exception("not LP")
        if self.Children != []:
            raise Exception("not leaf")
        prompt = prompt_to_prove_from_little_problem(self.Interior)
        proof = find_proof_little_smarter(bundle_proving,prompt,temperature=temperature,max_iter=max_iter)
        if isinstance(proof,str):
            self.Value = True
            self.Text = proof
            ancestor = self.Parent
            while ancestor != None:
                print("ok")
                if isinstance(ancestor.Interior,smashed_problem):
                    value = True
                    for i in ancestor.Children:
                        value = value and i.Value
                    ancestor.Value = value
                if isinstance(ancestor.Interior,LittleProblem):
                    value = False
                    for i in ancestor.Children:
                        value = value or i.Value
                    ancestor.Value = value
                ancestor = ancestor.Parent
            return True
        else:
            self.Smash_Leaf(bundle_smashing,deg)
            return False


    def TryNode(self,
                bundle_proving : NanoGPTBundle,
                bundle_smashing : NanoGPTBundle,
                temperature : float = 1.0,
                max_iter : int = 20,
                deg : int = 10):

        if self.Value == True:
            return True
        if isinstance(self.Interior,LittleProblem) and self.Children == []:
            return self.RunLeaf(bundle_proving,bundle_smashing,temperature=temperature,max_iter=max_iter,deg=deg)
        elif isinstance(self.Interior,LittleProblem):
            weights = []
            for i in self.Children:
                weights.append(i.Probability)
            next_step = random.choices(self.Children,weights=weights)[0]
            return next_step.TryNode(bundle_proving,bundle_smashing,temperature=temperature,max_iter=max_iter,deg=deg)
        elif isinstance(self.Interior,smashed_problem):
            for i in self.Children:
                if not i.TryNode(bundle_proving,bundle_smashing,temperature=temperature,max_iter=max_iter,deg=deg):
                    return False
            return True


    def to_str(self):
        ans = ""
        if isinstance(self.Interior,smashed_problem):
            ans += str(self.Interior)
        else:
            print(self.Text)
            ans += self.Text
        ans += "\n"
        if self.Children != [] and self.Interior == []:
            ans += "\t."
        else:
            for i in self.Children:
                little_ans = i.to_str()
                little_ans = "\t" + little_ans
                little_ans = little_ans.replace("\n","\n\t")
                ans += little_ans
        return ans

In [85]:
parse_infix("(a ∧ b) → (¬a → c)")

n = node(LittleProblem(Context([],[]),parse_infix("(a ∧ b) → (¬a → c)")),
    Parent = None,
    Children = [],
    Text = "",
    Value = False,
    Probability = 1.0)
n.TryNode(bundle_proving,bundle_smashing,temperature=1.0,max_new_lines=10,deg=1)
print("ok")
n.TryNode(bundle_proving,bundle_smashing,temperature=1.0,max_new_lines=10,deg=1)

1
Prove that: (a ∧ b) → (¬a → c)
1. ⊤   ⊤-introduction

1. ⊤   ⊤-introduction

2
Prove that: (a ∧ b) → (¬a → c)
1. ⊤   ⊤-introduction
2. ¬a   weakening, 1

1. ⊤   ⊤-introduction
2. ¬a   weakening, 1

3
Prove that: (a ∧ b) → (¬a → c)
1. ¬a    assumption

1. ¬a    assumption

4
Prove that: (a ∧ b) → (¬a → c)
1. ¬a    assumption
2. a ∧ b    assumption

1. ¬a    assumption
2. a ∧ b    assumption

5
Prove that: (a ∧ b) → (¬a → c)
1. ¬a    assumption
2. a ∧ b    assumption
3. a   ∧-elimination1, 2

1. ¬a    assumption
2. a ∧ b    assumption
3. a   ∧-elimination1, 2

6
Prove that: (a ∧ b) → (¬a → c)
1. ⊤   ⊤-introduction
2. ¬a   weakening, 1
3. a ∧ b    assumption

1. ⊤   ⊤-introduction
2. ¬a   weakening, 1
3. a ∧ b    assumption

7
Prove that: (a ∧ b) → (¬a → c)
1. ¬a    assumption
2. a ∧ b    assumption
3. a   ∧-elimination1, 2
4. ⊥   ¬-elimination, 1, 3

1. ¬a    assumption
2. a ∧ b    assumption
3. a   ∧-elimination1, 2
4. ⊥   ¬-elimination, 1, 3

8
Prove that: (a ∧ b) → (¬a → c)
1. ¬a   

False

In [86]:
proof = """Prove that: ¬((q ∧ m) ↔ ((i ∨ b) ∧ f)) → ¬¬a
assuming that: a
1. ¬¬p   assumption
2. p    ¬¬-elimination, 1
3. ¬¬p → p    →-introduction, 1–2
4. ¬((q ∧ m) ↔ ((i ∨ b) ∧ f))   assumption
5. (¬¬p → p) ∧ ¬((q ∧ m) ↔ ((i ∨ b) ∧ f))    ∧-introduction, 3, 4
6. ¬((q ∧ m) ↔ ((i ∨ b) ∧ f))    ∧-elimination2, 5
7. ¬((q ∧ m) ↔ ((i ∨ b) ∧ f))    ∧-elimination2, 5
8. a   weakening, 6
9. ¬a   assumption
10. ⊥    ¬-elimination, 9, 8
11. ¬¬((q ∧ m) ↔ ((i ∨ b) ∧ f))    ¬-introduction, 7–10
12. ⊥    ¬-elimination, 11, 7
13. a    RAA, 9–10
14. ¬a    ¬-introduction, 13–12
15. a   weakening, 7
16. ⊥    ¬-elimination, 9, 15
17. a    RAA, 14–16
18. ⊥    ¬-elimination, 9, 15
19. ¬a    ¬-introduction, 17–10
20. ¬¬a    ¬-introduction, 19–18
21. ¬((q ∧ m) ↔ ((i ∨ b) ∧ f)) → ¬¬a    →-introduction, 7–20"""
print(check_proof_with_assumptions(proof))

{1: 1 [] R0(a) ; ¬¬R0(p) ⊢ ¬¬R0(p) Assumption, 2: 2 [1] R0(a) ; ¬¬R0(p) ⊢ R0(p) NegationOfNegation, 3: 3 [1, 2] R0(a) ;   ⊢ ¬¬R0(p) → R0(p) ImplicationIntroduction, 4: 4 [] R0(a) ; ¬((R0(q) ∧ R0(m)) ↔ ((R0(i) ∨ R0(b)) ∧ R0(f))) ⊢ ¬((R0(q) ∧ R0(m)) ↔ ((R0(i) ∨ R0(b)) ∧ R0(f))) Assumption, 5: 5 [3, 4] R0(a) ; ¬((R0(q) ∧ R0(m)) ↔ ((R0(i) ∨ R0(b)) ∧ R0(f))) ⊢ (¬¬R0(p) → R0(p)) ∧ ¬((R0(q) ∧ R0(m)) ↔ ((R0(i) ∨ R0(b)) ∧ R0(f))) ConjunctionIntroduction, 6: 6 [5] R0(a) ; ¬((R0(q) ∧ R0(m)) ↔ ((R0(i) ∨ R0(b)) ∧ R0(f))) ⊢ ¬((R0(q) ∧ R0(m)) ↔ ((R0(i) ∨ R0(b)) ∧ R0(f))) ConjunctionElimination2, 7: 7 [5] R0(a) ; ¬((R0(q) ∧ R0(m)) ↔ ((R0(i) ∨ R0(b)) ∧ R0(f))) ⊢ ¬((R0(q) ∧ R0(m)) ↔ ((R0(i) ∨ R0(b)) ∧ R0(f))) ConjunctionElimination2, 8: 8 [6] R0(a) ; ¬((R0(q) ∧ R0(m)) ↔ ((R0(i) ∨ R0(b)) ∧ R0(f))) ⊢ R0(a) Weakening, 9: 9 [] R0(a) ; ¬R0(a) ⊢ ¬R0(a) Assumption, 10: 10 [9, 8] R0(a) ; ¬R0(a), ¬((R0(q) ∧ R0(m)) ↔ ((R0(i) ∨ R0(b)) ∧ R0(f))) ⊢ ⊥ NegationElimination, 11: 11 [7, 10] R0(a) ; ¬R0(a) ⊢ ¬¬((R0(q) ∧ R

In [88]:
parse_infix("(a ∧ b) → (¬a → c)")

n = node(LittleProblem(Context([],[]),parse_infix("(a ∧ b) → (¬a → c)")),
    Parent = None,
    Children = [],
    Text = "",
    Value = False,
    Probability = 1.0)
for i in range(20):
    print(i,"----------------------------------------------------------------------------------------")
    if n.TryNode(bundle_proving,bundle_smashing,temperature=1.0,max_new_lines=15,deg=5):
        break
    # coś się pętli (chyba przy aktualizowaniu Value)
    # napisz ładne printowanie

0 ----------------------------------------------------------------------------------------
1
Prove that: (a ∧ b) → (¬a → c)
1. ¬a   assumption

1. ¬a   assumption

2
Prove that: (a ∧ b) → (¬a → c)
1. ¬a   assumption
2. a ∧ b   assumption

1. ¬a   assumption
2. a ∧ b   assumption

3
Prove that: (a ∧ b) → (¬a → c)
1. ¬a   assumption
2. a ∧ b   assumption
3. a    ∧-elimination1, 2

1. ¬a   assumption
2. a ∧ b   assumption
3. a    ∧-elimination1, 2

4
Prove that: (a ∧ b) → (¬a → c)
1. ¬a   assumption
2. a ∧ b   assumption
3. a    ∧-elimination1, 2

1. ¬a   assumption
2. a ∧ b   assumption
3. a    ∧-elimination1, 2

5
Prove that: (a ∧ b) → (¬a → c)
1. ¬a   assumption
2. a ∧ b   assumption
3. a    ∧-elimination1, 2

1. ¬a   assumption
2. a ∧ b   assumption
3. a    ∧-elimination1, 2

6
Prove that: (a ∧ b) → (¬a → c)
1. ¬a   assumption
2. a ∧ b   assumption
3. a    ∧-elimination1, 2

1. ¬a   assumption
2. a ∧ b   assumption
3. a    ∧-elimination1, 2

7
Prove that: (a ∧ b) → (¬a → c)
1. ¬¬c   a

In [ ]:
to_infix_LP(n.Interior)
#for i in n.Children:
#    print(i.Interior)
#for i in n.Children[0].Children:
#    print(to_infix_LP(i.Interior))


for i in n.Children[0].Children[0].Children[0].Children[0].Children:
    for j in i.Children:
        for k in j.Children:
            for l in k.Children:
                for m in l.Children:
                    print(str(m.Interior))

In [5]:
def add_line(proof_text : str,line : str):# proof_text bez założeń
    def first_int(s):
        i = 0
        while i < len(s) and s[i].isdigit():
            i += 1
        return int(s[:i])
    import re

    def shift_numbers(line: str, n: int) -> str:
        pattern = r'(^|[^A-Za-z])(\d+)(?!\d)'

        def repl(match):
            prefix = match.group(1)
            num_str = match.group(2)
            x = int(num_str)
            return prefix + (str(x + 1) if x > n else num_str)
        return re.sub(pattern, repl, line)
    idx = first_int(line)

    proof_splited = proof_text.splitlines()
    ans_splited = []
    for i in proof_splited:
        if i[0] in ["1","2","3","4","5","6","7","8","9"]:
            if first_int(i) == idx:
                ans_splited.append(line)
            ans_splited.append(shift_numbers(i,idx))
        else:
            ans_splited.append(i)
    ans = "\n".join(ans_splited)
    return ans



def NormalizeProof(proof_text : str, new_base_context):
    proof_splitted = proof_text.splitlines()
    for i in range(len(proof_splitted)):
        if proof_splitted[i][0] == "1":
            idx = i
    proof_raw  = ""
    for i in range(idx,len(proof_splitted)):
        proof_raw += proof_splitted[i] + "\n"
    print(proof_raw)



proof_text ='''Prove that: ⊥
assuming that: ¬c,
¬a,
a ∧ b
1. a ∧ b    assumption
2. (a ∧ b) ∧ (a ∧ b)   ∧-introduction, 1, 1
3. a ∧ b   weakening, 2
4. ¬a    assumption
5. a   ∧-elimination1, 3
6. ⊥   ¬-elimination, 4, 5'''
print(add_line(proof_text,'2. ¬a    assumption'))

Prove that: ⊥
assuming that: ¬c,
¬a,
a ∧ b
1. a ∧ b    assumption
2. ¬a    assumption
2. (a ∧ b) ∧ (a ∧ b)   ∧-introduction, 1, 1
4. a ∧ b   weakening, 2
5. ¬a    assumption
6. a   ∧-elimination1, 4
7. ⊥   ¬-elimination, 5, 6
